> **Catatan versi multi-dataset.** Notebook ini menjalankan protokol locked-holdout yang sama pada beberapa dataset tabular. Pilih `DATASET_ID` di Sel 1 (`wine_red` mereproduksi angka paper). Protokol Sel 3–11 tidak berubah; hanya sumber data, definisi target biner/ordinal, dan rentang level ordinal yang menjadi parameter. Artefak per dataset ditulis ke `runs/<DATASET_ID>/`.

# Eksperimen Wine Quality: Klasifikasi Biner dan Analisis Ordinal

Notebook pendukung paper **Do Deep Tabular Models Outperform Classical Ensembles on Small Wine Quality Data? A Locked-Holdout Evaluation**, oleh Heriyanto dan Suharjito. Eksperimen utama membandingkan model klasik, MLP, transformer tabular, dan soft-voting untuk target `quality ≥ 6`. Analisis tambahan mempertahankan urutan nilai kualitas 3–8; batas kelas biner merupakan definisi tugas penelitian, bukan sertifikasi mutu wine.

## Cara membaca notebook

Nomor **Sel 1–11** merujuk pada sebelas tahap kode di bawah, bukan jumlah seluruh sel Markdown dan kode. Rujukan halaman mengikuti urutan **sembilan halaman PDF `HERIYANTO_PAPER.pdf`** yang diperiksa pada 7 September 2026; nomor halaman dapat berubah jika tata letak paper direvisi. Bagian III menjelaskan metode, sedangkan Bagian IV menyajikan hasil dan pembahasannya. Nama tabel dan gambar mengikuti paper tersebut.

Alur penelitian: audit dan deduplikasi → pembagian development/holdout → pencarian model dari prediksi out-of-fold (OOF) development → penguncian pilihan → fitting final dan evaluasi holdout → analisis statistik serta diagnostik. Sel 1–9 tidak memakai holdout untuk pemilihan model. Pemilihan epoch neural saat fitting final di Sel 10 tetap menggunakan pembagian internal development.

## Status eksekusi dan batas reproduksibilitas

Keluaran tersimpan berasal dari eksekusi pada PC peneliti, dengan arsip `archive/Wine Final Binary Ordinal Research - executed backup before warning cleanup 2026-09-06.ipynb`. Setelah eksekusi tersebut, source Sel 4, 6, 9, dan 10 mengalami penyesuaian kompatibilitas regularisasi scikit-learn, opsi nested tensor PyTorch, dan timestamp pandas. Kesetaraan numerik source setelah penyesuaian itu belum diverifikasi melalui eksekusi ulang. Penyuntingan dokumentasi ini hanya mengubah Markdown dan komentar; kode yang dijalankan serta keluaran historis dipertahankan.

Keluaran mencatat artefak dalam `wine_final_output` pada PC Windows peneliti. Folder run itu belum tersedia dalam direktori Wine lokal saat pemeriksaan 7 September 2026; CSV prediksi, lock, dan hash final belum dapat diverifikasi ulang dari salinan lokal. Pemuatan hasil lama membutuhkan artefak run aslinya. Untuk eksekusi berikutnya, simpan notebook yang benar-benar dijalankan, environment, dan artefaknya sebagai satu run yang dapat ditelusuri.

Evaluasi ini merupakan validasi internal setelah eksplorasi dataset yang sama, bukan validasi eksternal: split lama dengan seed 42 pernah diperiksa sebelum protokol seed `20260906`. Eksperimen KAN enam kelas terdahulu berada dalam arsip dan bukan metode utama paper. Penggunaan bantuan AI dijelaskan dalam *Acknowledgment of AI Use* pada halaman 9. Ide visual diimplementasikan ulang mengikuti split dan lock penelitian ini; keluarannya tidak disalin sebagai hasil eksperimen.


## 1. Lingkungan komputasi dan parameter protokol

**Rujukan paper:** III.C, halaman 2; III.E, halaman 3; III.K (*Implementation and Reproducibility*), halaman 5.

Sel ini menetapkan konteks eksekusi sebelum data dibaca: seed `20260906`, holdout 20%, lima fold development, serta inner-validation 18% untuk pemilihan epoch neural. Mode `research` memakai anggaran yang dilaporkan dalam paper: 114 konfigurasi klasik, 16 neural, dan 42 ordinal pada keluarga model yang tersedia dalam eksekusi tersebut. Mode `quick` hanya untuk memeriksa alur, bukan menghasilkan angka pembanding paper.

Jumlah thread estimator dibatasi maksimal 16 dan fold dijalankan serial. PyTorch memakai CUDA bila tersedia; versi pustaka dan perangkat dicatat agar perbedaan lingkungan dapat ditelusuri. Tidak ada instalasi paket otomatis. CatBoost dan XGBoost bersifat opsional dan tidak tersedia pada eksekusi yang dilaporkan; menjalankan notebook dengan pustaka tersebut dapat menambah kandidat dan mengubah konteks perbandingan.

Registry `DATASETS` mendaftar seluruh dataset yang memakai protokol ini; `DATASET_ID` memilih satu di antaranya dan `wine_red` mereproduksi angka paper. Entri berpasangan `*_efreq` / `*_ewidth` adalah eksperimen keseimbangan terkendali: target kontinu yang sama dipotong menjadi enam bin frekuensi-sama (seimbang) atau enam bin lebar-sama (timpang seperti grade wine). `PROJECT_ROOT` dideteksi dari folder yang memuat `Dataset/`; `DATA_PATH` dapat menunjuk CSV secara langsung. Artefak run ditempatkan di `runs/<DATASET_ID>/`, sehingga cache, lock, dan hasil antar-dataset tidak pernah bercampur dan pencarian yang terhenti dapat dilanjutkan per dataset. Nilai `ALLOW_FINAL_RERUN=False` mempertahankan pemeriksaan pemuatan hasil lengkap; recovery atau rerun harus diaktifkan secara eksplisit dan diberi penanda oleh kode Sel 10.

Anggaran komputasi disetel untuk mesin eksekusi (Intel i5-14500, 20 logical processor, 64 GB RAM, RTX 3080): `N_THREADS` memakai 18 thread dan menyisakan dua untuk OS serta kernel Jupyter, sedangkan `CV_N_JOBS=1` mempertahankan fold serial. TF32 dan `cudnn.benchmark` diaktifkan untuk GPU Ampere. Untuk menjalankan banyak dataset sekaligus, gunakan `run_all_datasets.py` di folder proyek.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 1/11 DIMULAI — Konfigurasi, dependensi, dan perangkat keras', flush=True)
try:
    import os, sys, math, json, hashlib, random, warnings, gc, inspect
    from pathlib import Path

    # ============================================================================
    # ANGGARAN KOMPUTASI  (disetel untuk Intel i5-14500 / 20 logical / 64 GB / RTX 3080)
    # ----------------------------------------------------------------------------
    # N_THREADS : thread untuk estimator sklearn yang mendukung n_jobs
    #             (RandomForest, ExtraTrees, HistGradientBoosting). Menyisakan 2
    #             logical processor untuk OS + kernel Jupyter.
    # CV_N_JOBS : 1 = fold berjalan serial (sesuai teks paper). Menaikkannya ke 5
    #             menjalankan kelima fold paralel — hasil numerik TIDAK berubah
    #             (sklearn menjamin n_jobs tidak memengaruhi hasil), hanya lebih
    #             cepat untuk model bergaya single-thread (LogisticRegression,
    #             RBF-SVM). Bila dinaikkan, perbarui kalimat "folds ran serially"
    #             pada Bagian III.K paper.
    # Pengungkit terbesar bukan thread, melainkan menjalankan beberapa DATASET_ID
    # sekaligus di proses terpisah — lihat run_all_datasets.py di folder Paper.
    # ============================================================================
    CPU_COUNT = os.cpu_count() or 1
    N_THREADS = max(1, min(18, CPU_COUNT - 2 if CPU_COUNT > 2 else 1))
    CV_N_JOBS = 1
    os.environ.setdefault('OMP_NUM_THREADS', str(N_THREADS))
    os.environ.setdefault('MKL_NUM_THREADS', str(N_THREADS))
    os.environ.setdefault('NUMEXPR_NUM_THREADS', str(N_THREADS))
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

    import importlib.util
    required = ['numpy', 'pandas', 'sklearn', 'scipy', 'matplotlib', 'joblib']
    missing = [name for name in required if importlib.util.find_spec(name) is None]
    if missing:
        raise ImportError('Paket wajib belum tersedia: ' + ', '.join(missing) + '. Notebook tidak memasang paket otomatis.')

    import numpy as np
    import pandas as pd
    import sklearn
    import scipy
    import matplotlib
    import matplotlib.pyplot as plt
    import joblib
    from IPython.display import display

    TORCH_AVAILABLE = importlib.util.find_spec('torch') is not None
    CATBOOST_AVAILABLE = importlib.util.find_spec('catboost') is not None
    XGBOOST_AVAILABLE = importlib.util.find_spec('xgboost') is not None
    # TabPFN [5]: transformer yang dipralatih pada tugas sintetis. Dirancang
    # TANPA tuning, sehingga anggaran kandidatnya sengaja kecil — ini perbedaan
    # sifat model, bukan kelalaian pencarian; nyatakan demikian di paper.
    TABPFN_AVAILABLE = importlib.util.find_spec('tabpfn') is not None

    TORCH_IMPORT_ERROR = None
    if TORCH_AVAILABLE:
        try:
            import torch
            import torch.nn as nn
            import torch.nn.functional as F
            torch.set_num_threads(N_THREADS)
            # TF32 pada Ampere (RTX 3080): matmul/conv lebih cepat, presisi cukup
            # untuk jaringan kecil ini. cudnn.benchmark menguntungkan karena bentuk
            # batch tetap sepanjang training.
            torch.set_float32_matmul_precision('high')
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            torch.backends.cudnn.benchmark = True
            DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        except Exception as exc:
            TORCH_AVAILABLE = False
            TORCH_IMPORT_ERROR = f'{type(exc).__name__}: {exc}'
            torch = nn = F = None
            DEVICE = 'unavailable'
    else:
        torch = nn = F = None
        DEVICE = 'unavailable'

    SEED = 20260906
    MODE = 'research'                 # Tabel II, hlm. 3: anggaran kandidat paper memakai mode research.
    PROTOCOL_VERSION = 'tabular-locked-holdout-v3-multidataset'

    # ============================================================================
    # PEMILIHAN DATASET
    # ----------------------------------------------------------------------------
    # Protokol Sel 3-11 tidak berubah antar-dataset: hanya sumber data, definisi
    # target biner/ordinal, dan rentang level ordinal yang menjadi parameter.
    # Setel DATASET_ID ke salah satu kunci DATASETS lalu jalankan Sel 1-11.
    # Setiap dataset menulis artefak ke runs/<DATASET_ID>/... sehingga cache dan
    # lock tidak tercampur; "training continue" bekerja per-dataset.
    #
    # binning target kontinu: 'quantile' = 6 bin frekuensi-sama (seimbang);
    # 'uniform' = 6 bin lebar-sama (timpang, meniru grade wine). Pasangan
    # efreq/ewidth adalah eksperimen terkendali keseimbangan kelas.
    # ============================================================================
    DATASET_ID = 'wine_red'

    _ORD_NATIVE = {'kind': 'native'}
    _ORD_EFREQ = {'kind': 'bin', 'bins': 6, 'strategy': 'quantile'}
    _ORD_EWIDTH = {'kind': 'bin', 'bins': 6, 'strategy': 'uniform'}
    _SUBSAMPLE = 4898  # samakan n dengan wine putih untuk dataset besar

    DATASETS = {
        # --- wine: grade asli 3-8 / 3-9 (kasus timpang kanonik paper) ---
        'wine_red':   dict(file='winequality-red.csv',   sep=';', target='quality',
                           ordinal=_ORD_NATIVE, binary='ge:6'),
        'wine_white': dict(file='winequality-white.csv', sep=';', target='quality',
                           ordinal=_ORD_NATIVE, binary='ge:6'),
        # Kontrol seimbang untuk wine: grade diganti bin frekuensi-sama, tetapi
        # target BINER sengaja tetap 'quality >= 6' agar hanya definisi ordinal
        # yang berubah terhadap wine_red / wine_white — perbandingan terkendali.
        'wine_red_efreq':   dict(file='winequality-red.csv',   sep=';', target='quality',
                                 ordinal=_ORD_EFREQ, binary='ge:6'),
        'wine_white_efreq': dict(file='winequality-white.csv', sep=';', target='quality',
                                 ordinal=_ORD_EFREQ, binary='ge:6'),
        # --- 2025: kualitas udara (OpenML 46880, unggah 2025-04) ---
        # Analog langsung wine: pengukuran fisikokimia -> grade kualitas berjenjang.
        # Level 3=Good, 2=Moderate, 1=Poor, 0=Hazardous (2000/1500/1000/500).
        'airquality': dict(file='air_quality_2025.csv', sep=',', target='air_quality_level',
                           ordinal=_ORD_NATIVE, binary='ge:2'),
        # --- 2025: Miami housing, dikurasi untuk TabArena (OpenML 46942, unggah 2025-04) ---
        'miami_efreq':  dict(file='miami_housing_2025.csv', sep=',', target='SALE_PRC',
                             subsample=_SUBSAMPLE, ordinal=_ORD_EFREQ, binary='median'),
        'miami_ewidth': dict(file='miami_housing_2025.csv', sep=',', target='SALE_PRC',
                             subsample=_SUBSAMPLE, ordinal=_ORD_EWIDTH, binary='median'),
        # --- LENGAN A/B KEBOCORAN: identik dengan entri di atas, hanya dedup dimatikan.
        #     Selisihnya = inflasi akibat duplikat yang menyeberang train/test.
        'wine_red_nodedup':   dict(file='winequality-red.csv',   sep=';', target='quality',
                                   ordinal=_ORD_NATIVE, binary='ge:6', dedup=False),
        'wine_white_nodedup': dict(file='winequality-white.csv', sep=';', target='quality',
                                   ordinal=_ORD_NATIVE, binary='ge:6', dedup=False),
        'concrete_efreq_nodedup': dict(file='concrete_compressive_strength.csv', sep=',',
                                       target='compressive_strength',
                                       ordinal=_ORD_EFREQ, binary='median', dedup=False),
        'superconduct_efreq_nodedup': dict(file='superconductivity.csv', sep=',',
                                           target='critical_temp', subsample=_SUBSAMPLE,
                                           ordinal=_ORD_EFREQ, binary='median', dedup=False),
        # --- target kontinu: pasangan efreq / ewidth ---
        'concrete_efreq':  dict(file='concrete_compressive_strength.csv', sep=',',
                                target='compressive_strength',
                                ordinal=_ORD_EFREQ, binary='median'),
        'concrete_ewidth': dict(file='concrete_compressive_strength.csv', sep=',',
                                target='compressive_strength',
                                ordinal=_ORD_EWIDTH, binary='median'),
        'energy_efreq':  dict(file='energy_efficiency.csv', sep=',', target='Y1', drop=['Y2'],
                              ordinal=_ORD_EFREQ, binary='median'),
        'energy_ewidth': dict(file='energy_efficiency.csv', sep=',', target='Y1', drop=['Y2'],
                              ordinal=_ORD_EWIDTH, binary='median'),
        'california_efreq':  dict(file='california_housing.csv', sep=',', target='MedHouseVal',
                                  subsample=_SUBSAMPLE, ordinal=_ORD_EFREQ, binary='median'),
        'california_ewidth': dict(file='california_housing.csv', sep=',', target='MedHouseVal',
                                  subsample=_SUBSAMPLE, ordinal=_ORD_EWIDTH, binary='median'),
        'abalone_efreq':  dict(file='abalone.csv', sep=',', target='Rings', drop=['Sex'],
                               ordinal=_ORD_EFREQ, binary='median'),
        'abalone_ewidth': dict(file='abalone.csv', sep=',', target='Rings', drop=['Sex'],
                               ordinal=_ORD_EWIDTH, binary='median'),
        'gasturbine_efreq':  dict(file='gas_turbine_emission.csv', sep=',', target='NOX', drop=['CO'],
                                  subsample=_SUBSAMPLE, ordinal=_ORD_EFREQ, binary='median'),
        'gasturbine_ewidth': dict(file='gas_turbine_emission.csv', sep=',', target='NOX', drop=['CO'],
                                  subsample=_SUBSAMPLE, ordinal=_ORD_EWIDTH, binary='median'),
        'superconduct_efreq':  dict(file='superconductivity.csv', sep=',', target='critical_temp',
                                    subsample=_SUBSAMPLE, ordinal=_ORD_EFREQ, binary='median'),
        'superconduct_ewidth': dict(file='superconductivity.csv', sep=',', target='critical_temp',
                                    subsample=_SUBSAMPLE, ordinal=_ORD_EWIDTH, binary='median'),
    }
    if DATASET_ID not in DATASETS:
        raise ValueError(f"DATASET_ID={DATASET_ID!r} tidak dikenal. Pilihan: {sorted(DATASETS)}")
    DATASET_SPEC = DATASETS[DATASET_ID]

    DATA_PATH = None                  # isi Path CSV bila ingin menimpa lokasi file dataset terpilih.
    PROJECT_ROOT_OVERRIDE = None      # None memakai deteksi folder proyek.

    def detect_project_root(override=None):
        roots = []
        if override is not None:
            roots.append(Path(override).expanduser())
        env_root = os.environ.get('PAPER_PROJECT_ROOT') or os.environ.get('WINE_PROJECT_ROOT')
        if env_root:
            roots.append(Path(env_root).expanduser())
        home, cwd = Path.home(), Path.cwd()
        roots += [
            cwd, cwd.parent, cwd.parent.parent,
            home / 'Documents' / 'Paper',
            home / 'OneDrive' / 'Documents' / 'Paper',
            home / 'OneDrive' / 'Desktop' / 'Paper',
        ]
        seen = set()
        for root in roots:
            try:
                root = root.resolve()
            except OSError:
                continue
            if root in seen:
                continue
            seen.add(root)
            if (root / 'Dataset').is_dir() or (root / 'data').is_dir():
                return root
        return Path(override).expanduser().resolve() if override is not None else cwd.resolve()

    PROJECT_ROOT = detect_project_root(PROJECT_ROOT_OVERRIDE)
    DATASET_DIR = PROJECT_ROOT / 'Dataset'
    if not DATASET_DIR.is_dir() and (PROJECT_ROOT / 'data').is_dir():
        DATASET_DIR = PROJECT_ROOT / 'data'
    OUTPUT_ROOT = PROJECT_ROOT / 'runs' / DATASET_ID
    TEST_SIZE = 0.20
    CV_SPLITS = 5
    INNER_STOP_SIZE = 0.18
    RARE_LEVEL_MIN = 2 * CV_SPLITS    # level ordinal dengan < 10 observasi bersih digabung ke tetangga (Sel 2)
    RUN_NEURAL = True
    RUN_ORDINAL = True
    RUN_OPTIONAL_BOOSTERS = True
    RUN_INTERPRETABILITY = True       # III.J/Gambar 6, hlm. 5/8: importance pada held-out development folds.
    ALLOW_FINAL_RERUN = False         # III.K, hlm. 5: recovery/rerun memerlukan aktivasi eksplisit dan penanda run.
    PRIMARY_SELECTION_METRIC = 'roc_auc'

    if MODE not in {'quick', 'research'}:
        raise ValueError("MODE harus 'quick' atau 'research'.")
    if PRIMARY_SELECTION_METRIC != 'roc_auc':
        raise ValueError("Protokol ini mengunci PRIMARY_SELECTION_METRIC='roc_auc'. Ubah versi protokol bila metrik utama berubah.")

    CLASSIC_LIMITS = ({
        'LogisticRegression': 6, 'SVM_RBF': 8, 'RandomForest': 6,
        'ExtraTrees': 6, 'HistGradientBoosting': 8, 'XGBoost': 6, 'CatBoost': 6, 'TabPFN': 2,
    } if MODE == 'quick' else {
        'LogisticRegression': 18, 'SVM_RBF': 28, 'RandomForest': 18,
        'ExtraTrees': 22, 'HistGradientBoosting': 28, 'XGBoost': 22, 'CatBoost': 22, 'TabPFN': 3,
    })
    NEURAL_LIMIT_PER_FAMILY = 2 if MODE == 'quick' else 8
    NEURAL_MAX_EPOCHS = 80 if MODE == 'quick' else 320
    NEURAL_PATIENCE = 15 if MODE == 'quick' else 35
    ENSEMBLE_WEIGHT_TRIALS = 100 if MODE == 'quick' else 800
    BOOTSTRAP_REPEATS = 500 if MODE == 'quick' else 3000
    PERMUTATION_REPEATS = 4 if MODE == 'quick' else 20

    warnings.filterwarnings('ignore', category=FutureWarning)
    pd.set_option('display.max_columns', 80)
    pd.set_option('display.width', 180)
    np.set_printoptions(precision=5, suppress=True)
    try:
        plt.style.use('seaborn-v0_8-whitegrid')
    except OSError:
        plt.style.use('default')

    random.seed(SEED)
    np.random.seed(SEED)
    if TORCH_AVAILABLE:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

    print(f'Python executable : {sys.executable}')
    print(f'Python version    : {sys.version.split()[0]}')
    print(f'numpy/pandas      : {np.__version__} / {pd.__version__}')
    print(f'scikit-learn     : {sklearn.__version__}')
    print(f'Mode             : {MODE}')
    print(f'Dataset ID       : {DATASET_ID}  (file={DATASET_SPEC["file"]}, ordinal={DATASET_SPEC["ordinal"]["kind"]})')
    print(f'Project root     : {PROJECT_ROOT}')
    print(f'Dataset dir      : {DATASET_DIR}')
    print(f'Output root      : {OUTPUT_ROOT}')
    print(f'CPU threads      : estimator={N_THREADS}, outer-CV jobs={CV_N_JOBS}')
    if TORCH_AVAILABLE:
        print(f'PyTorch          : {torch.__version__}')
        print(f'Device neural    : {DEVICE}')
        if torch.cuda.is_available():
            print(f'GPU              : {torch.cuda.get_device_name(0)}')
    else:
        detail = f' ({TORCH_IMPORT_ERROR})' if TORCH_IMPORT_ERROR else ''
        print(f'⚠ PyTorch tidak tersedia/bermasalah: MLP/FT-Transformer dilewati{detail}.')
    print(f'CatBoost         : {"tersedia" if CATBOOST_AVAILABLE else "tidak tersedia — dilewati"}')
    print(f'XGBoost          : {"tersedia" if XGBOOST_AVAILABLE else "tidak tersedia — dilewati"}')
    print(f'TabPFN           : {"tersedia" if TABPFN_AVAILABLE else "tidak tersedia — dilewati"}')
    print(f'Interpretability : {"aktif" if RUN_INTERPRETABILITY else "nonaktif"} | permutation repeats={PERMUTATION_REPEATS}')
    print('Tidak ada perintah pip/install di notebook ini.')
except KeyboardInterrupt:
    print('⏸ SEL 1/11 DIHENTIKAN OLEH PENGGUNA — Konfigurasi, dependensi, dan perangkat keras', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 1/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 1/11 SELESAI — Konfigurasi, dependensi, dan perangkat keras ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 2. Audit sumber data dan penghapusan observasi duplikat

**Rujukan paper:** III.B–C dan Tabel I, halaman 2; *Data Availability*, halaman 9.

Sel ini memuat dataset yang ditunjuk `DATASET_ID`. Untuk `wine_red`, sumbernya adalah subset red wine Wine Quality UCI dengan 11 fitur fisikokimia dan label `quality` 3–8. Loader membaca CSV, menyeragamkan nama kolom, membuang kolom yang didaftarkan pada `drop`, memeriksa kelengkapan serta nilai numerik yang valid, dan menerapkan subsampel berseed bila dataset terlalu besar. Pemeriksaan tidak mengganti data hilang dengan nilai buatan: kondisi data yang tidak memenuhi syarat menghentikan alur untuk diperiksa.

Audit duplikat menggunakan seluruh vektor 11 fitur. Jika fitur identik memiliki label berbeda, eksperimen dihentikan; jika label konsisten, satu observasi dipertahankan. Penghapusan dilakukan **sebelum split** untuk mencegah salinan observasi identik berada di development dan holdout sekaligus. Pada data yang dilaporkan, 240 baris duplikat dihapus dari 1599 baris sehingga tersisa 1359 observasi.

Dua target dibentuk dari data bersih yang sama. `Y_BINARY` mengikuti aturan `binary` pada registry: `ge:6` menghasilkan `quality ≥ 6` seperti paper, sedangkan `median` membelah target kontinu pada mediannya. `Y_QUALITY` menyimpan level ordinal — grade asli untuk `ordinal='native'` (wine, air quality) atau kode bin 0..K-1 untuk target kontinu. Level yang lebih langka dari `RARE_LEVEL_MIN = 2 × CV_SPLITS` digabung ke tetangganya agar StratifiedKFold lima fold tetap sah; pada wine merah tidak ada level yang tergabung sehingga perilakunya identik dengan paper, sedangkan pada wine putih grade 9 (5 observasi) bergabung ke grade 8. Definisi target adalah definisi tugas penelitian, bukan transform yang dipelajari, sehingga dihitung dari data bersih penuh — sama seperti ambang `quality ≥ 6`. `DATA_AUDIT` mencatat jumlah observasi, distribusi label, dan hash sumber/data bersih untuk pemeriksaan identitas run pada tahap berikutnya. Tidak ada penyeimbangan kelas melalui resampling pada tahap ini.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 2/11 DIMULAI — Memuat dan mengaudit dataset', flush=True)
try:
    def canonical_column(name):
        s = str(name).strip().lower()
        for ch in ['(', ')', '[', ']', '{', '}', ',', '^', '%', '/', '\\', '.', '-', ':', ';', '#']:
            s = s.replace(ch, ' ')
        return '_'.join(s.split())

    def locate_dataset(spec, explicit=None):
        candidates = []
        if explicit is not None:
            candidates.append(Path(explicit).expanduser())
        env_path = os.environ.get('PAPER_DATA_PATH') or os.environ.get('WINE_DATA_PATH')
        if env_path:
            candidates.append(Path(env_path).expanduser())
        fname = spec['file']
        cwd = Path.cwd()
        for base in [DATASET_DIR, PROJECT_ROOT / 'Dataset', PROJECT_ROOT / 'data',
                     cwd / 'Dataset', cwd / 'data', cwd, cwd.parent / 'Dataset']:
            candidates.append(base / fname)
        for candidate in candidates:
            if candidate.is_file():
                return candidate.resolve()
        checked = '\n'.join(f'  - {p}' for p in candidates)
        raise FileNotFoundError(f'{fname} tidak ditemukan. Path yang diperiksa:\n' + checked)

    def _resolve_name(requested, columns):
        stripped = {str(c).strip(): c for c in columns}
        canon = {canonical_column(c): c for c in columns}
        if requested in stripped:
            return stripped[requested]
        if str(requested).strip() in stripped:
            return stripped[str(requested).strip()]
        if canonical_column(requested) in canon:
            return canon[canonical_column(requested)]
        raise KeyError(f'Kolom {requested!r} tidak ada. Tersedia: {list(columns)}')

    def read_dataset_frame(path, spec):
        frame = pd.read_csv(path, sep=spec.get('sep', ','))
        if frame.shape[1] == 1:
            frame = pd.read_csv(path, sep=None, engine='python')
        target_src = _resolve_name(spec['target'], frame.columns)
        drop_src = [_resolve_name(d, frame.columns) for d in spec.get('drop', [])]
        keep = [c for c in frame.columns if c not in set(drop_src)]
        frame = frame[keep].copy()
        rename = {c: canonical_column(c) for c in frame.columns}
        frame = frame.rename(columns=rename)
        target_col = rename[target_src]
        feature_cols = [c for c in frame.columns if c != target_col]
        if not feature_cols:
            raise ValueError('Tidak ada kolom fitur setelah drop/target.')
        return frame[feature_cols + [target_col]].copy(), feature_cols, target_col

    def _stratify_key(levels, min_count):
        """Gabung level ordinal dengan < min_count observasi ke tetangga terdekat.
        Menjaga StratifiedKFold(CV_SPLITS) valid. Untuk wine-red tidak ada level
        yang tergabung (perilaku identik paper)."""
        levels = np.asarray(levels)
        order = sorted(np.unique(levels).tolist())
        counts = {v: int(np.sum(levels == v)) for v in order}
        mapping = {v: v for v in order}
        changed = True
        while changed and len(set(mapping.values())) > 2:
            changed = False
            groups = sorted(set(mapping.values()))
            sizes = {g: sum(counts[v] for v in order if mapping[v] == g) for g in groups}
            small = [g for g in groups if sizes[g] < min_count]
            if not small:
                break
            g = small[0]
            gi = groups.index(g)
            neighbour = groups[gi + 1] if gi == 0 else groups[gi - 1]
            for v in order:
                if mapping[v] == g:
                    mapping[v] = neighbour
            changed = True
        remap = {g: i for i, g in enumerate(sorted(set(mapping.values())))}
        return np.array([remap[mapping[v]] for v in levels], dtype=np.int64), mapping

    def build_targets(target_values, spec):
        """Kembalikan (ordinal_int_consecutive, binary_int, meta).
        Definisi target adalah definisi tugas penelitian, bukan transform yang
        dipelajari; dihitung dari data bersih penuh, sama seperti 'quality >= 6'."""
        raw = np.asarray(target_values, dtype=float)
        cfg = spec['ordinal']
        meta = {'ordinal_kind': cfg['kind']}
        if cfg['kind'] == 'native':
            q = np.rint(raw).astype(np.int64)
            if not np.allclose(raw, q, atol=1e-9):
                raise ValueError('ordinal="native" tetapi target bukan bilangan bulat.')
            _, mapping = _stratify_key(q, RARE_LEVEL_MIN)
            merged = {int(k): int(v) for k, v in mapping.items() if k != v}
            if merged:
                print(f'  ⚠ Level ordinal langka digabung untuk menjaga CV: {merged}', flush=True)
            # pertahankan nilai grade asli (mis. wine 3..8); hanya level yang
            # digabung yang berpindah ke tetangganya.
            ordinal = np.array([int(mapping[int(v)]) for v in q], dtype=np.int64)
            meta['native_levels'] = sorted(np.unique(q).tolist())
            meta['merged_levels'] = merged
            # Penggabungan level di TENGAH dapat menyisakan lubang (mis. {1,2,4,5}).
            # Model ordinal kumulatif mensyaratkan level runut, jadi petakan ulang
            # sambil mempertahankan urutan. Untuk wine tidak ada lubang sehingga
            # grade 3..8 dipertahankan apa adanya.
            survivors = sorted(np.unique(ordinal).tolist())
            if survivors != list(range(survivors[0], survivors[-1] + 1)):
                renumber = {v: survivors[0] + i for i, v in enumerate(survivors)}
                print(f'  ⚠ Level tersisa tidak runut {survivors}; dipetakan ulang: {renumber}', flush=True)
                ordinal = np.array([renumber[int(v)] for v in ordinal], dtype=np.int64)
                meta['renumbered_levels'] = {int(k): int(v) for k, v in renumber.items()}
        else:
            bins = int(cfg['bins'])
            if cfg['strategy'] == 'quantile':
                codes = pd.qcut(raw, bins, labels=False, duplicates='drop')
            else:
                codes = pd.cut(raw, bins, labels=False, include_lowest=True)
            codes = np.asarray(codes, dtype=float)
            if np.isnan(codes).any():
                raise ValueError('Binning menghasilkan NaN; periksa target.')
            uniq = sorted(np.unique(codes).tolist())
            relabel = {u: i for i, u in enumerate(uniq)}
            ordinal = np.array([relabel[c] for c in codes], dtype=np.int64)
            # gabung bin langka (equal-width bisa menghasilkan bin sangat tipis)
            key2, mapping2 = _stratify_key(ordinal, RARE_LEVEL_MIN)
            if len(np.unique(key2)) != len(np.unique(ordinal)):
                merged2 = {int(k): int(v) for k, v in mapping2.items() if k != v}
                print(f'  ⚠ Bin ordinal langka digabung: {merged2}', flush=True)
                ordinal = key2
            meta['bins_requested'] = bins
            meta['bins_effective'] = int(len(np.unique(ordinal)))

        rule = spec['binary']
        if rule.startswith('ge:'):
            thr = float(rule.split(':', 1)[1])
            binary = (raw >= thr).astype(np.int64)
            meta['binary_rule'] = f'target >= {thr:g}'
        elif rule == 'median':
            med = float(np.median(raw))
            binary = (raw > med).astype(np.int64)
            meta['binary_rule'] = f'target > median ({med:.6g})'
        else:
            raise ValueError(f'binary rule tidak dikenal: {rule}')
        if len(np.unique(binary)) < 2:
            raise ValueError('Target biner hanya memiliki satu kelas.')
        return ordinal, binary, meta

    # --- muat dataset terpilih ---
    DATA_FILE = locate_dataset(DATASET_SPEC, DATA_PATH)
    SOURCE_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
    frame_full, FEATURE_COLUMNS, TARGET_RAW_COLUMN = read_dataset_frame(DATA_FILE, DATASET_SPEC)

    for col in frame_full.columns:
        frame_full[col] = pd.to_numeric(frame_full[col], errors='coerce')
    if frame_full.isna().any().any():
        n_bad = int(frame_full.isna().sum().sum())
        raise ValueError(f'Ditemukan {n_bad} nilai missing/non-numerik pada fitur/target.')
    if not np.isfinite(frame_full[FEATURE_COLUMNS].to_numpy(dtype=float)).all():
        raise ValueError('Fitur mengandung nilai non-finite.')

    subsample_n = DATASET_SPEC.get('subsample')
    subsample_applied = 0
    if subsample_n and len(frame_full) > subsample_n:
        frame_full = frame_full.sample(n=int(subsample_n), random_state=SEED).sort_index().reset_index(drop=True)
        subsample_applied = int(subsample_n)

    df_raw = frame_full.reset_index(drop=True).copy()
    ORD_RAW, BIN_RAW, TARGET_META = build_targets(df_raw[TARGET_RAW_COLUMN].to_numpy(), DATASET_SPEC)
    df_raw = df_raw.assign(ordinal_level=ORD_RAW, binary_target=BIN_RAW)

    strict_conflict = (DATASET_SPEC['ordinal']['kind'] == 'native')
    conflicts = df_raw.groupby(FEATURE_COLUMNS, dropna=False)['ordinal_level'].nunique()
    conflict_count = int((conflicts > 1).sum())
    if conflict_count and strict_conflict:
        raise ValueError(f'Ada {conflict_count} vektor fitur identik dengan label ordinal berbeda; audit manual diperlukan.')
    if conflict_count:
        print(f'  ⚠ {conflict_count} grup fitur identik berlabel berbeda (target kontinu); baris pertama dipertahankan.', flush=True)

    # III.B: deduplikasi SEBELUM split adalah default protokol. dedup=False
    # sengaja MEMBIARKAN vektor identik menyeberang batas train/test — dipakai
    # hanya untuk mengukur besarnya inflasi akibat kebocoran duplikat (A/B).
    DEDUP = bool(DATASET_SPEC.get('dedup', True))
    duplicate_mask = df_raw.duplicated(subset=FEATURE_COLUMNS, keep='first')
    if DEDUP:
        df = df_raw.loc[~duplicate_mask].reset_index(drop=True)
    else:
        print(f'  ⚠ DEDUPLIKASI DIMATIKAN — {int(duplicate_mask.sum())} vektor identik '
              f'dibiarkan dan dapat menyeberang train/test (lengan A/B kebocoran).', flush=True)
        df = df_raw.reset_index(drop=True)

    TARGET_COLUMN = 'ordinal_level'
    Y_QUALITY = df['ordinal_level'].to_numpy(dtype=np.int64)
    Y_BINARY = df['binary_target'].to_numpy(dtype=np.int64)
    X_ALL = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    ORDINAL_LEVELS = np.array(sorted(np.unique(Y_QUALITY).tolist()), dtype=np.int64)
    ORDINAL_MIN = int(ORDINAL_LEVELS.min())
    ORDINAL_MAX = int(ORDINAL_LEVELS.max())
    if not np.array_equal(ORDINAL_LEVELS, np.arange(ORDINAL_MIN, ORDINAL_MAX + 1)):
        raise RuntimeError(f'Level ordinal tidak berurutan: {ORDINAL_LEVELS.tolist()}')
    if len(ORDINAL_LEVELS) < 2:
        raise RuntimeError('Target ordinal hanya memiliki satu level setelah pemrosesan.')

    canonical_bytes = df[FEATURE_COLUMNS + ['ordinal_level', 'binary_target']].to_csv(
        index=False, float_format='%.17g', lineterminator='\n').encode('utf-8')
    CLEAN_SHA256 = hashlib.sha256(canonical_bytes).hexdigest()

    DATA_AUDIT = {
        'dataset_id': DATASET_ID, 'spec': json.loads(json.dumps(DATASET_SPEC, default=str)),
        'source_path': str(DATA_FILE), 'source_sha256': SOURCE_SHA256,
        'clean_sha256': CLEAN_SHA256, 'rows_raw': int(len(df_raw)),
        'rows_clean': int(len(df)), 'duplicates_removed': int(duplicate_mask.sum()),
        'subsample_applied': subsample_applied, 'dedup': DEDUP,
        'conflicting_feature_groups': conflict_count,
        'target_column_source': TARGET_RAW_COLUMN, 'target_meta': TARGET_META,
        'ordinal_levels': [int(v) for v in ORDINAL_LEVELS],
        'ordinal_counts_clean': {str(k): int(v) for k, v in pd.Series(Y_QUALITY).value_counts().sort_index().items()},
        'binary_counts_clean': {str(k): int(v) for k, v in pd.Series(Y_BINARY).value_counts().sort_index().items()},
        'features': FEATURE_COLUMNS, 'n_features': len(FEATURE_COLUMNS),
    }

    print(f'Dataset          : {DATASET_ID}  <-  {DATA_FILE.name}')
    print(f'Fitur ({len(FEATURE_COLUMNS)})       : {FEATURE_COLUMNS}')
    print(f'Target sumber     : {TARGET_RAW_COLUMN}  | {TARGET_META.get("binary_rule")}')
    if subsample_applied:
        print(f'Subsample        : {subsample_applied} baris (seed {SEED})')
    print(f'Bentuk mentah    : {df_raw.shape}')
    print(f'Duplikat dibuang : {int(duplicate_mask.sum())}')
    print(f'Bentuk bersih    : {df.shape}')
    print(f'Level ordinal    : {ORDINAL_LEVELS.tolist()}  ({TARGET_META.get("ordinal_kind")})')
    print('Distribusi ordinal:', DATA_AUDIT['ordinal_counts_clean'])
    print('Target biner      :', DATA_AUDIT['binary_counts_clean'], f'(0/1 via {TARGET_META.get("binary_rule")})')
    print(f'Clean SHA256      : {CLEAN_SHA256}')
    display(df.head())
except KeyboardInterrupt:
    print('⏸ SEL 2/11 DIHENTIKAN OLEH PENGGUNA — Memuat dan mengaudit dataset', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 2/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 2/11 SELESAI — Memuat dan mengaudit dataset ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 3. Pembagian data terkunci dan eksplorasi development

**Rujukan paper:** III.C, Tabel I, dan Gambar 1, halaman 2; III.J–K, IV.A, dan Gambar 2, halaman 5.

Data bersih dibagi 80:20 menjadi development dan holdout menggunakan seed `20260906`; untuk `wine_red` pembagiannya 1087 dan 272 seperti paper. Stratifikasi mengikuti **level ordinal**, bukan hanya target biner, agar kelas ekstrem tetap terwakili. Indeks dan identitas pembagian disimpan dalam `split_indices.npz` serta `split_lock.json`. Pemeriksaan ulang memastikan pembagian mencakup data yang sesuai dan tidak saling tumpang tindih.

Di dalam development dibentuk dua susunan lima fold yang dibekukan: `BINARY_CV_FOLDS` distratifikasi menurut target biner, sedangkan `ORDINAL_CV_FOLDS` menurut kualitas asli. Susunan fold disimpan dalam `cv_folds.npz`. Data, protokol, paket, dan fold dihubungkan melalui hash konteks evaluasi; cache dari konteks berbeda tidak boleh dianggap sebagai hasil run yang sama.

Pada tahap ini hanya `X_DEV`, `Y_DEV`, dan `Q_DEV` yang disiapkan sebagai array untuk pemodelan. Array holdout baru disiapkan setelah pemeriksaan lock di Sel 10. Pemisahan ini membatasi penggunaan holdout dalam alur eksperimen, bukan berarti peneliti belum pernah mengeksplorasi dataset sumber.

Ringkasan fitur dan grafik `development_eda_overview.png` menjadi dasar Gambar 2. EDA memakai development saja dan bersifat deskriptif; hasilnya tidak digunakan untuk menghapus fitur, memilih transformasi berdasarkan holdout, atau menetapkan hubungan sebab-akibat.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 3/11 DIMULAI — Membuat atau memuat split terkunci', flush=True)
try:
    from sklearn.model_selection import train_test_split, StratifiedKFold

    # III.J, hlm. 5: permutation importance hanya diagnostik (Gambar 6), bukan
    # bagian seleksi. Biayanya 5 fold x repeats x n_fitur prediksi RandomForest;
    # pada dataset berdimensi tinggi (superconductivity: 81 fitur) 20 repeat
    # berarti 8100 prediksi dan mendominasi waktu run. Turunkan HANYA pada kasus
    # itu; seluruh dataset <= 30 fitur (termasuk wine) tidak berubah.
    if len(FEATURE_COLUMNS) > 30 and PERMUTATION_REPEATS > 5:
        print(f'Permutation repeats {PERMUTATION_REPEATS} -> 5 karena dataset punya '
              f'{len(FEATURE_COLUMNS)} fitur (diagnostik, tidak memengaruhi seleksi).')
        PERMUTATION_REPEATS = 5

    SETTINGS_PAYLOAD = {
        'dataset_id': DATASET_ID,
        'dataset_spec': json.loads(json.dumps(DATASET_SPEC, default=str)),
        'n_features': len(FEATURE_COLUMNS),
        'mode': MODE, 'seed': SEED, 'test_size': TEST_SIZE, 'cv_splits': CV_SPLITS,
        'inner_stop_size': INNER_STOP_SIZE, 'run_neural': RUN_NEURAL,
        'run_ordinal': RUN_ORDINAL, 'run_optional_boosters': RUN_OPTIONAL_BOOSTERS,
        'run_interpretability': RUN_INTERPRETABILITY, 'permutation_repeats': PERMUTATION_REPEATS,
        'torch_available': TORCH_AVAILABLE, 'catboost_available': CATBOOST_AVAILABLE,
        'xgboost_available': XGBOOST_AVAILABLE, 'tabpfn_available': TABPFN_AVAILABLE,
        'classic_limits': CLASSIC_LIMITS,
        'neural_limit_per_family': NEURAL_LIMIT_PER_FAMILY,
        'neural_max_epochs': NEURAL_MAX_EPOCHS, 'neural_patience': NEURAL_PATIENCE,
        'primary_selection_metric': PRIMARY_SELECTION_METRIC,
    }
    SETTINGS_HASH = hashlib.sha256(json.dumps(SETTINGS_PAYLOAD, sort_keys=True).encode()).hexdigest()
    RUN_DIR = OUTPUT_ROOT / f'{PROTOCOL_VERSION}_{CLEAN_SHA256[:8]}_{SETTINGS_HASH[:8]}'
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / 'cache' / 'binary_classic').mkdir(parents=True, exist_ok=True)
    (RUN_DIR / 'cache' / 'binary_neural').mkdir(parents=True, exist_ok=True)
    (RUN_DIR / 'cache' / 'ordinal').mkdir(parents=True, exist_ok=True)
    (RUN_DIR / 'models').mkdir(parents=True, exist_ok=True)

    with open(RUN_DIR / 'data_audit.json', 'w', encoding='utf-8') as f:
        json.dump(DATA_AUDIT, f, indent=2)

    split_file = RUN_DIR / 'split_indices.npz'
    split_lock_file = RUN_DIR / 'split_lock.json'
    if split_file.exists() != split_lock_file.exists():
        raise RuntimeError('Artefak split tidak lengkap. Hapus split_indices.npz dan split_lock.json pada run ini, lalu ulangi Sel 3.')

    split_loaded = split_file.exists() and split_lock_file.exists()
    if split_loaded:
        lock = json.loads(split_lock_file.read_text(encoding='utf-8'))
        expected_lock = {
            'protocol': PROTOCOL_VERSION, 'clean_sha256': CLEAN_SHA256, 'seed': SEED,
            'test_size': TEST_SIZE, 'cv_splits': CV_SPLITS, 'settings_hash': SETTINGS_HASH,
        }
        mismatch = {k: (lock.get(k), v) for k, v in expected_lock.items() if lock.get(k) != v}
        if mismatch:
            raise RuntimeError(f'Split tersimpan tidak cocok dengan konfigurasi aktif: {mismatch}')
        with np.load(split_file) as packed:
            DEV_IDX = packed['development'].astype(np.int64)
            TEST_IDX = packed['test'].astype(np.int64)
        print('Split terkunci ditemukan dan digunakan kembali.')
    else:
        indices = np.arange(len(df), dtype=np.int64)
        DEV_IDX, TEST_IDX = train_test_split(
            indices, test_size=TEST_SIZE, random_state=SEED,
            shuffle=True, stratify=Y_QUALITY,
        )

    if DEV_IDX.ndim != 1 or TEST_IDX.ndim != 1:
        raise RuntimeError('Indeks split harus berupa array satu dimensi.')
    combined = np.concatenate([DEV_IDX, TEST_IDX])
    if not np.array_equal(np.sort(combined), np.arange(len(df), dtype=np.int64)):
        raise RuntimeError('Split harus merupakan permutasi tepat indeks 0..n-1 tanpa duplikat, indeks negatif, atau baris hilang.')
    expected_test_rows = int(math.ceil(TEST_SIZE * len(df)))
    expected_dev_rows = len(df) - expected_test_rows
    if len(DEV_IDX) != expected_dev_rows or len(TEST_IDX) != expected_test_rows:
        raise RuntimeError(
            f'Ukuran split salah: development/test={len(DEV_IDX)}/{len(TEST_IDX)}, '
            f'seharusnya {expected_dev_rows}/{expected_test_rows}.'
        )
    split_hasher = hashlib.sha256()
    for label, values in [('development', DEV_IDX), ('test', TEST_IDX)]:
        array = np.ascontiguousarray(values, dtype=np.int64)
        split_hasher.update(label.encode('utf-8'))
        split_hasher.update(np.asarray(array.shape, dtype=np.int64).tobytes())
        split_hasher.update(array.tobytes())
    SPLIT_INDICES_SHA256 = split_hasher.hexdigest()

    if split_loaded:
        if lock.get('development_rows') != len(DEV_IDX) or lock.get('test_rows') != len(TEST_IDX):
            raise RuntimeError('Ukuran indeks split berbeda dari split lock.')
        if lock.get('indices_sha256') != SPLIT_INDICES_SHA256:
            raise RuntimeError('Hash indeks split tidak cocok; artefak split mungkin berubah/rusak.')
    else:
        split_tmp = split_file.with_name(split_file.name + '.tmp.npz')
        np.savez_compressed(split_tmp, development=DEV_IDX, test=TEST_IDX)
        os.replace(split_tmp, split_file)
        split_lock = {
            'protocol': PROTOCOL_VERSION, 'clean_sha256': CLEAN_SHA256,
            'seed': SEED, 'test_size': TEST_SIZE, 'cv_splits': CV_SPLITS,
            'settings_hash': SETTINGS_HASH, 'indices_sha256': SPLIT_INDICES_SHA256,
            'stratified_on': f'ordinal_levels_{ORDINAL_MIN}_to_{ORDINAL_MAX}',
            'development_rows': int(len(DEV_IDX)), 'test_rows': int(len(TEST_IDX)),
            'rule': 'Test may be accessed only in Cell 10 after selection locks are written.',
        }
        split_lock_tmp = split_lock_file.with_name(split_lock_file.name + '.tmp')
        split_lock_tmp.write_text(json.dumps(split_lock, indent=2), encoding='utf-8')
        os.replace(split_lock_tmp, split_lock_file)
        print('Split baru dibuat dan dikunci.')

    # III.C, hlm. 2: bentuk array development saja; array holdout menunggu validasi lock di Sel 10.
    X_DEV = X_ALL[DEV_IDX]
    Y_DEV = Y_BINARY[DEV_IDX]
    Q_DEV = Y_QUALITY[DEV_IDX]

    bin_skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED + 11)
    ord_skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED + 29)
    BINARY_CV_FOLDS = [(tr.astype(np.int64), va.astype(np.int64)) for tr, va in bin_skf.split(X_DEV, Y_DEV)]
    ORDINAL_CV_FOLDS = [(tr.astype(np.int64), va.astype(np.int64)) for tr, va in ord_skf.split(X_DEV, Q_DEV)]

    fold_payload = {}
    for i, (tr, va) in enumerate(BINARY_CV_FOLDS):
        fold_payload[f'bin_train_{i}'] = tr; fold_payload[f'bin_valid_{i}'] = va
    for i, (tr, va) in enumerate(ORDINAL_CV_FOLDS):
        fold_payload[f'ord_train_{i}'] = tr; fold_payload[f'ord_valid_{i}'] = va
    np.savez_compressed(RUN_DIR / 'cv_folds.npz', **fold_payload)

    from importlib.metadata import version as distribution_version, PackageNotFoundError
    def optional_version(distribution):
        try: return distribution_version(distribution)
        except PackageNotFoundError: return None
    package_versions = {
        'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__, 'scipy': scipy.__version__,
        'matplotlib': matplotlib.__version__, 'joblib': optional_version('joblib'),
        'torch': getattr(torch, '__version__', None) if TORCH_AVAILABLE else None,
        'catboost': optional_version('catboost') if CATBOOST_AVAILABLE else None,
        'xgboost': optional_version('xgboost') if XGBOOST_AVAILABLE else None,
        'tabpfn': optional_version('tabpfn') if TABPFN_AVAILABLE else None,
    }
    context_hasher = hashlib.sha256()
    context_hasher.update(PROTOCOL_VERSION.encode())
    context_hasher.update(CLEAN_SHA256.encode())
    context_hasher.update(SETTINGS_HASH.encode())
    context_hasher.update(json.dumps(package_versions, sort_keys=True).encode())
    for key in sorted(fold_payload):
        context_hasher.update(key.encode()); context_hasher.update(np.ascontiguousarray(fold_payload[key]).tobytes())
    EVALUATION_CONTEXT_HASH = context_hasher.hexdigest()
    (RUN_DIR / 'evaluation_context.json').write_text(json.dumps({
        'evaluation_context_hash': EVALUATION_CONTEXT_HASH,
        'settings': SETTINGS_PAYLOAD, 'package_versions': package_versions,
    }, indent=2), encoding='utf-8')

    # IV.A/Gambar 2, hlm. 5: EDA development bersifat deskriptif, bukan dasar penghapusan fitur.
    development_frame = df.iloc[DEV_IDX][FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
    development_target = pd.Series(Y_BINARY[DEV_IDX], index=development_frame.index).astype(np.int64)
    feature_skewness = development_frame[FEATURE_COLUMNS].skew()
    corr_quality = development_frame[FEATURE_COLUMNS].corrwith(development_frame[TARGET_COLUMN])
    corr_binary = development_frame[FEATURE_COLUMNS].corrwith(development_target)
    EDA_SUMMARY = pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'skewness': feature_skewness.reindex(FEATURE_COLUMNS).to_numpy(),
        'pearson_quality': corr_quality.reindex(FEATURE_COLUMNS).to_numpy(),
        'pearson_binary_target': corr_binary.reindex(FEATURE_COLUMNS).to_numpy(),
    })
    EDA_SUMMARY.to_csv(RUN_DIR / 'development_eda_feature_summary.csv', index=False)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    quality_counts = development_frame[TARGET_COLUMN].value_counts().reindex(list(ORDINAL_LEVELS), fill_value=0)
    axes[0, 0].bar(quality_counts.index.astype(str), quality_counts.values, color='#4C78A8', edgecolor='white')
    for x, value in enumerate(quality_counts.values):
        axes[0, 0].text(x, value + max(quality_counts.max() * .015, 1), str(int(value)), ha='center', fontsize=9)
    axes[0, 0].set(title=f'Distribusi level ordinal — development ({DATASET_ID})', xlabel='Level ordinal', ylabel='Jumlah sampel')

    binary_counts = development_target.value_counts().reindex([0, 1], fill_value=0)
    axes[0, 1].bar(['quality < 6', 'quality ≥ 6'], binary_counts.values,
                   color=['#E45756', '#54A24B'], edgecolor='white')
    for x, value in enumerate(binary_counts.values):
        axes[0, 1].text(x, value + max(binary_counts.max() * .015, 1), str(int(value)), ha='center', fontsize=9)
    axes[0, 1].set(title='Target biner — development', ylabel='Jumlah sampel')

    skew_plot = feature_skewness.sort_values()
    axes[1, 0].barh(skew_plot.index, skew_plot.values,
                    color=['#E45756' if value < 0 else '#4C78A8' for value in skew_plot.values])
    axes[1, 0].axvline(0, color='black', linewidth=.8)
    axes[1, 0].set(title='Skewness fitur — alasan menguji scaler robust/power/quantile', xlabel='Skewness')

    corr_plot = corr_binary.sort_values()
    axes[1, 1].barh(corr_plot.index, corr_plot.values,
                    color=['#E45756' if value < 0 else '#54A24B' for value in corr_plot.values])
    axes[1, 1].axvline(0, color='black', linewidth=.8)
    axes[1, 1].set(title='Korelasi Pearson dengan target biner — deskriptif', xlabel='Korelasi')

    fig.suptitle(f'{DATASET_ID}: EDA Development-Only (n={len(development_frame)})', fontsize=15, y=1.01)
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'development_eda_overview.png', dpi=180, bbox_inches='tight')
    plt.show()
    strongest = corr_binary.abs().sort_values(ascending=False).head(3).index.tolist()
    print('✅ Grafik EDA development-only tersimpan: development_eda_overview.png')
    print('Tiga korelasi biner absolut terbesar (deskriptif):', strongest)

    TEST_ACCESS_STARTED_PATH = RUN_DIR / 'TEST_ACCESS_STARTED.json'
    FINAL_COMPLETE_PATH = RUN_DIR / 'FINAL_COMPLETE.json'
    TEST_ACCESSED = TEST_ACCESS_STARTED_PATH.exists() or FINAL_COMPLETE_PATH.exists()
    print(f'Run directory : {RUN_DIR}')
    print(f'Development   : {len(DEV_IDX)} baris | binary counts={np.bincount(Y_DEV).tolist()}')
    if FINAL_COMPLETE_PATH.exists():
        test_status = 'evaluasi final sudah tersimpan; Sel 10 akan memuat hasil'
    elif TEST_ACCESS_STARTED_PATH.exists():
        test_status = 'proses final sebelumnya pernah dimulai tetapi belum lengkap'
    else:
        test_status = 'belum disentuh; label tidak dipakai untuk pemilihan'
    print(f'Locked test   : {len(TEST_IDX)} baris | {test_status}')
    print(f'CV biner      : {CV_SPLITS} fold tetap')
    print(f'CV ordinal    : {CV_SPLITS} fold tetap')
except KeyboardInterrupt:
    print('⏸ SEL 3/11 DIHENTIKAN OLEH PENGGUNA — Membuat atau memuat split terkunci', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 3/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 3/11 SELESAI — Membuat atau memuat split terkunci ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 4. Evaluator OOF, preprocessing, dan ruang pencarian model klasik

**Rujukan paper:** III.D, halaman 2–3; Tabel II dan III.F, halaman 3; III.H, halaman 4.

Sel ini mendefinisikan kandidat logistic regression, RBF-SVM, Random Forest, Extra Trees, dan histogram-based gradient boosting beserta prosedur evaluasinya. Anggaran per keluarga pada mode penelitian berturut-turut adalah 18, 28, 18, 22, dan 28 konfigurasi. Kandidat berasal dari konfigurasi anchor dan bagian grid yang diacak secara reproducible; anggaran berbeda antarkeluarga, sehingga perbandingan tidak menyamakan biaya komputasi.

Preprocessing yang diperlukan ditempatkan dalam pipeline estimator dan dipelajari ulang hanya dari training fold. Prediksi probabilitas untuk setiap observasi development berasal dari model yang tidak melatih observasi tersebut pada fold bersangkutan. Probabilitas OOF digabung untuk menghitung ROC-AUC, macro-F1, balanced accuracy, dan metrik pendamping.

`choose_threshold` menguji 241 ambang dari 0.20 hingga 0.80 dengan langkah 0.0025. Pemilihan memaksimalkan macro-F1; jika seri, digunakan balanced accuracy, lalu kedekatan ambang ke 0.5. Pemeringkatan **konfigurasi model** berbeda: ROC-AUC didahulukan, kemudian macro-F1 dan balanced accuracy. Threshold hanya dipilih dari development, bukan holdout.

Cache kandidat memuat konfigurasi, probabilitas OOF, metrik, waktu fitting, dan signature konteks evaluasi. Hash mencegah cache dengan data atau pengaturan berbeda tercampur. Probabilitas dibatasi ke interval aman di fungsi metrik untuk menghindari masalah numerik pada log loss.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 4/11 DIMULAI — Menyiapkan model klasik dan evaluasi OOF', flush=True)
try:
    from sklearn.base import BaseEstimator, clone
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
    from sklearn.model_selection import ParameterGrid, cross_val_predict
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
        average_precision_score, precision_score, recall_score, matthews_corrcoef,
        log_loss, brier_score_loss, confusion_matrix, cohen_kappa_score,
        mean_absolute_error,
    )

    if CATBOOST_AVAILABLE:
        try:
            from catboost import CatBoostClassifier, CatBoostRegressor
        except Exception as exc:
            CATBOOST_AVAILABLE = False
            print(f'⚠ CatBoost terdeteksi tetapi gagal diimpor; dilewati: {type(exc).__name__}: {exc}', flush=True)
    if XGBOOST_AVAILABLE:
        try:
            from xgboost import XGBClassifier, XGBRegressor
        except Exception as exc:
            XGBOOST_AVAILABLE = False
            print(f'⚠ XGBoost terdeteksi tetapi gagal diimpor; dilewati: {type(exc).__name__}: {exc}', flush=True)

    if TABPFN_AVAILABLE:
        try:
            import os as _os
            _os.environ.setdefault('TABPFN_NO_BROWSER', '1')  # jangan buka browser saat run batch
            from tabpfn import TabPFNClassifier, TabPFNRegressor
            TABPFN_DEVICE = 'cuda' if (TORCH_AVAILABLE and torch.cuda.is_available()) else 'cpu'
        except Exception as exc:
            TABPFN_AVAILABLE = False
            print(f'⚠ TabPFN terdeteksi tetapi gagal diimpor; dilewati: {type(exc).__name__}: {exc}', flush=True)

    def json_safe(value):
        if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
        if isinstance(value, np.ndarray): return value.tolist()
        if isinstance(value, (np.integer,)): return int(value)
        if isinstance(value, (np.floating,)): return float(value)
        if isinstance(value, Path): return str(value)
        return value

    def slug(text):
        return ''.join(ch if ch.isalnum() else '_' for ch in str(text)).strip('_').lower()

    def atomic_write_json(path, payload):
        path = Path(path); tmp = path.with_name(path.name + '.tmp')
        tmp.write_text(json.dumps(json_safe(payload), indent=2), encoding='utf-8')
        tmp.replace(path)

    def atomic_savez(path, **arrays):
        path = Path(path); tmp = path.with_name(path.name + '.tmp')
        with tmp.open('wb') as handle:
            np.savez_compressed(handle, **arrays)
        tmp.replace(path)

    def atomic_write_csv(frame, path):
        path = Path(path); tmp = path.with_name(path.name + '.tmp')
        frame.to_csv(tmp, index=False)
        tmp.replace(path)

    def make_scaler(name):
        if name == 'standard': return StandardScaler()
        if name == 'robust': return RobustScaler(quantile_range=(10, 90))
        if name == 'power': return PowerTransformer(method='yeo-johnson', standardize=True)
        if name == 'quantile':
            return QuantileTransformer(n_quantiles=256, output_distribution='normal', random_state=SEED)
        raise ValueError(f'Scaler tidak dikenal: {name}')

    def choose_threshold(y_true, probability):
        # III.F, hlm. 3: maksimum macro-F1 OOF; seri diputus balanced accuracy, lalu jarak ke 0.5.
        best = None
        for threshold in np.linspace(0.20, 0.80, 241):
            pred = (probability >= threshold).astype(np.int64)
            f1m = f1_score(y_true, pred, average='macro', zero_division=0)
            bacc = balanced_accuracy_score(y_true, pred)
            key = (f1m, bacc, -abs(threshold - 0.5))
            if best is None or key > best[0]:
                best = (key, float(threshold))
        return best[1]

    def binary_metrics(y_true, probability, threshold=0.5):
        probability = np.clip(np.asarray(probability, dtype=float), 1e-7, 1-1e-7)
        pred = (probability >= float(threshold)).astype(np.int64)
        cm = confusion_matrix(y_true, pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return {
            'threshold': float(threshold),
            'roc_auc': float(roc_auc_score(y_true, probability)),
            'average_precision': float(average_precision_score(y_true, probability)),
            'accuracy': float(accuracy_score(y_true, pred)),
            'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
            'f1_macro': float(f1_score(y_true, pred, average='macro', zero_division=0)),
            'f1_positive': float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
            'precision': float(precision_score(y_true, pred, zero_division=0)),
            'recall_sensitivity': float(recall_score(y_true, pred, zero_division=0)),
            'specificity': float(tn / max(tn + fp, 1)),
            'mcc': float(matthews_corrcoef(y_true, pred)),
            'log_loss': float(log_loss(y_true, np.c_[1-probability, probability], labels=[0, 1])),
            'brier': float(brier_score_loss(y_true, probability)),
        }

    def _candidate_subset(anchors, grid, limit, seed):
        rng = np.random.default_rng(seed)
        all_cfg = list(ParameterGrid(grid))
        rng.shuffle(all_cfg)
        merged, seen = [], set()
        for cfg in list(anchors) + all_cfg:
            key = json.dumps(json_safe(cfg), sort_keys=True)
            if key not in seen:
                seen.add(key); merged.append(cfg)
            if len(merged) >= limit: break
        return merged

    def classic_candidate_sets():
        sets = {}
        sets['LogisticRegression'] = _candidate_subset(
            [{'C': 1.0, 'penalty': 'l2', 'class_weight': None, 'scaler': 'standard'},
             {'C': 10.0, 'penalty': 'l2', 'class_weight': None, 'scaler': 'power'}],
            {'C': [0.01, .03, .1, .3, 1., 3., 10., 30., 100.],
             'penalty': ['l1', 'l2'], 'class_weight': [None, 'balanced'],
             'scaler': ['standard', 'robust', 'power']},
            CLASSIC_LIMITS['LogisticRegression'], SEED+101)
        sets['SVM_RBF'] = _candidate_subset(
            [{'C': 10.0, 'gamma': 'scale', 'class_weight': None, 'scaler': 'standard'},
             {'C': 3.0, 'gamma': .1, 'class_weight': None, 'scaler': 'power'},
             {'C': 30.0, 'gamma': .03, 'class_weight': 'balanced', 'scaler': 'standard'}],
            {'C': [.1, .3, 1., 3., 10., 30., 100., 300.],
             'gamma': ['scale', .003, .01, .03, .1, .3, 1.],
             'class_weight': [None, 'balanced'], 'scaler': ['standard', 'robust', 'power']},
            CLASSIC_LIMITS['SVM_RBF'], SEED+102)
        sets['RandomForest'] = _candidate_subset(
            [{'n_estimators': 800, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'class_weight': None}],
            {'n_estimators': [400, 800, 1200], 'max_depth': [None, 8, 14, 24],
             'max_features': ['sqrt', .5, .8, 1.0], 'min_samples_leaf': [1, 2, 3, 5],
             'class_weight': [None, 'balanced', 'balanced_subsample']},
            CLASSIC_LIMITS['RandomForest'], SEED+103)
        sets['ExtraTrees'] = _candidate_subset(
            [{'n_estimators': 800, 'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'class_weight': None}],
            {'n_estimators': [400, 800, 1200], 'max_depth': [None, 8, 14, 24],
             'max_features': ['sqrt', .5, .8, 1.0], 'min_samples_leaf': [1, 2, 3, 5],
             'class_weight': [None, 'balanced']},
            CLASSIC_LIMITS['ExtraTrees'], SEED+104)
        sets['HistGradientBoosting'] = _candidate_subset(
            [{'learning_rate': .06, 'max_iter': 600, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': .1, 'class_weight': None}],
            {'learning_rate': [.02, .04, .06, .08, .12], 'max_iter': [300, 600, 1000],
             'max_leaf_nodes': [7, 15, 31, 63], 'min_samples_leaf': [10, 20, 30, 40],
             'l2_regularization': [0., .01, .1, 1., 10.], 'class_weight': [None, 'balanced']},
            CLASSIC_LIMITS['HistGradientBoosting'], SEED+105)
        if RUN_OPTIONAL_BOOSTERS and XGBOOST_AVAILABLE:
            sets['XGBoost'] = _candidate_subset(
                [{'n_estimators': 600, 'max_depth': 3, 'learning_rate': .04, 'min_child_weight': 3,
                  'subsample': .85, 'colsample_bytree': .9, 'reg_alpha': .0, 'reg_lambda': 2.0}],
                {'n_estimators': [300, 600, 900], 'max_depth': [2, 3, 4, 5],
                 'learning_rate': [.02, .04, .07, .1], 'min_child_weight': [1, 3, 5, 10],
                 'subsample': [.7, .85, 1.0], 'colsample_bytree': [.7, .85, 1.0],
                 'reg_alpha': [0., .01, .1, 1.], 'reg_lambda': [.5, 1., 2., 5., 10.]},
                CLASSIC_LIMITS['XGBoost'], SEED+106)
        if RUN_OPTIONAL_BOOSTERS and CATBOOST_AVAILABLE:
            sets['CatBoost'] = _candidate_subset(
                [{'iterations': 800, 'depth': 6, 'learning_rate': .04, 'l2_leaf_reg': 3., 'random_strength': .5, 'auto_class_weights': None}],
                {'iterations': [400, 800, 1200], 'depth': [4, 5, 6, 7, 8],
                 'learning_rate': [.02, .04, .07, .1], 'l2_leaf_reg': [1., 3., 5., 10.],
                 'random_strength': [0., .5, 1.], 'auto_class_weights': [None, 'Balanced']},
                CLASSIC_LIMITS['CatBoost'], SEED+107)
        if TABPFN_AVAILABLE:
            # Tanpa hyperparameter tuning: hanya jumlah anggota ensemble internal.
            sets['TabPFN'] = _candidate_subset(
                [{'n_estimators': 4}, {'n_estimators': 8}, {'n_estimators': 2}],
                {'n_estimators': [2, 4, 8]},
                CLASSIC_LIMITS['TabPFN'], SEED+108)
        return sets

    def logistic_regularization_kwargs(penalty):
        """Gunakan API regulasi LogisticRegression yang sesuai versi sklearn."""
        parameter = inspect.signature(LogisticRegression).parameters.get('penalty')
        if parameter is None or parameter.default == 'deprecated':
            mapping = {'l1': 1.0, 'l2': 0.0}
            if penalty not in mapping:
                raise ValueError(f'Regularisasi logistic tidak didukung: {penalty!r}')
            return {'l1_ratio': mapping[penalty]}
        return {'penalty': penalty}

    def make_classic_estimator(family, cfg):
        cfg = dict(cfg)
        if family == 'LogisticRegression':
            scaler = make_scaler(cfg.pop('scaler'))
            regularization = logistic_regularization_kwargs(cfg.pop('penalty'))
            return Pipeline([('scale', scaler), ('model', LogisticRegression(
                solver='liblinear', max_iter=6000, random_state=SEED,
                **regularization, **cfg))])
        if family == 'SVM_RBF':
            scaler = make_scaler(cfg.pop('scaler'))
            return Pipeline([('scale', scaler), ('model', SVC(
                kernel='rbf', probability=True, cache_size=2048, random_state=SEED, **cfg))])
        if family == 'RandomForest':
            return RandomForestClassifier(random_state=SEED, n_jobs=N_THREADS, **cfg)
        if family == 'ExtraTrees':
            return ExtraTreesClassifier(random_state=SEED, n_jobs=N_THREADS, **cfg)
        if family == 'HistGradientBoosting':
            if 'class_weight' not in inspect.signature(HistGradientBoostingClassifier).parameters:
                cfg.pop('class_weight', None)
            return HistGradientBoostingClassifier(random_state=SEED, early_stopping=False, **cfg)
        if family == 'TabPFN':
            # fitur mentah (TabPFN menormalkan sendiri), sama seperti model tree.
            return TabPFNClassifier(device=TABPFN_DEVICE, ignore_pretraining_limits=True,
                                    memory_saving_mode=True, random_state=SEED, **cfg)
        if family == 'XGBoost':
            return XGBClassifier(tree_method='hist', eval_metric='logloss', random_state=SEED, n_jobs=N_THREADS, **cfg)
        if family == 'CatBoost':
            auto = cfg.pop('auto_class_weights', None)
            if auto is not None: cfg['auto_class_weights'] = auto
            return CatBoostClassifier(loss_function='Logloss', verbose=False, allow_writing_files=False,
                                      random_seed=SEED, thread_count=N_THREADS, task_type='CPU', **cfg)
        raise ValueError(f'Family tidak dikenal: {family}')

    def evaluate_classic_candidate(family, cfg, position, total):
        signature = hashlib.sha256((EVALUATION_CONTEXT_HASH + family +
                                    json.dumps(json_safe(cfg), sort_keys=True)).encode()).hexdigest()[:16]
        stem = f'{slug(family)}_{signature}'
        npz_path = RUN_DIR / 'cache' / 'binary_classic' / f'{stem}.npz'
        json_path = RUN_DIR / 'cache' / 'binary_classic' / f'{stem}.json'
        if npz_path.exists() and json_path.exists():
            try:
                record = json.loads(json_path.read_text(encoding='utf-8'))
                with np.load(npz_path) as packed:
                    probability = packed['probability'].copy()
                if (record.get('signature') == signature and len(probability) == len(Y_DEV)
                        and np.isfinite(probability).all()):
                    print(f'  ✓ {position}/{total} {family} cache | AUC={record["roc_auc"]:.4f}', flush=True)
                    return record, probability
                raise ValueError('isi cache tidak cocok')
            except Exception as cache_error:
                print(f'  ⚠ Cache {family} rusak/tidak cocok; dihitung ulang: {cache_error}', flush=True)
                npz_path.unlink(missing_ok=True); json_path.unlink(missing_ok=True)
        print(f'  ▶ {position}/{total} {family} kandidat dimulai', flush=True)
        started = _cell_time.perf_counter()
        try:
            estimator = make_classic_estimator(family, cfg)
            probability = cross_val_predict(
                estimator, X_DEV, Y_DEV, cv=BINARY_CV_FOLDS,
                method='predict_proba', n_jobs=CV_N_JOBS,
                pre_dispatch=CV_N_JOBS,
            )[:, 1]
            threshold = choose_threshold(Y_DEV, probability)
            record = {'family': family, 'config': json_safe(cfg),
                      **binary_metrics(Y_DEV, probability, threshold),
                      'fit_seconds': _cell_time.perf_counter()-started,
                      'signature': signature, 'status': 'ok'}
            atomic_savez(npz_path, probability=probability)
            atomic_write_json(json_path, record)
            print(f'  ✅ {family} | AUC={record["roc_auc"]:.4f} | macro-F1={record["f1_macro"]:.4f} | {record["fit_seconds"]:.1f}s', flush=True)
            return record, probability
        except Exception as exc:
            failure = {'family': family, 'config': json_safe(cfg), 'status': 'failed',
                       'error_type': type(exc).__name__, 'error': str(exc)}
            atomic_write_json(json_path, failure)
            print(f'  ⚠ {family} kandidat gagal dan dilewati: {type(exc).__name__}: {exc}', flush=True)
            return None, None

    CLASSIC_CANDIDATES = classic_candidate_sets()
    print('Kandidat klasik siap:')
    for family, candidates in CLASSIC_CANDIDATES.items():
        print(f'  {family:22s}: {len(candidates)} kandidat × {CV_SPLITS} fold')
except KeyboardInterrupt:
    print('⏸ SEL 4/11 DIHENTIKAN OLEH PENGGUNA — Menyiapkan model klasik dan evaluasi OOF', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 4/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 4/11 SELESAI — Menyiapkan model klasik dan evaluasi OOF ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 5. Evaluasi kandidat klasik pada lima fold development

**Rujukan paper:** Tabel II, halaman 3; IV.A dan Gambar 3, halaman 5; kolom development pada Tabel III, halaman 6.

Sel ini menjalankan evaluator Sel 4 untuk seluruh kandidat klasik. Hasil tiap kandidat disimpan sehingga pencarian yang terinterupsi dapat dilanjutkan dari cache yang cocok. Konfigurasi terbaik setiap keluarga dipilih menurut ROC-AUC OOF, dengan macro-F1 dan balanced accuracy sebagai pemecah seri. Konfigurasi dan probabilitasnya diteruskan ke tahap penguncian serta pembentukan ensemble di Sel 8.

`binary_classic_development_leaderboard.csv` memuat perbandingan seluruh kandidat; `binary_family_best_development.csv` dan `binary_classic_best.json` merangkum pilihan per keluarga. Angka pada tahap ini adalah **skor tuning/seleksi development**, bukan hasil holdout dan bukan estimasi nested cross-validation yang bebas dari pengaruh pemilihan konfigurasi.

Baseline prior memberikan probabilitas konstan sebesar proporsi kelas positif development, dengan threshold 0.5. Baseline tersebut hanya konteks pembanding, tidak diikutkan sebagai pemenang atau anggota ensemble. Skor development baseline bersifat deskriptif karena prior dihitung dari seluruh development, bukan dihitung ulang per training fold.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 5/11 DIMULAI — Pencarian model klasik pada development set', flush=True)
try:
    CLASSIC_ROWS = []
    CLASSIC_PROB_BY_SIGNATURE = {}
    total_candidates = sum(len(v) for v in CLASSIC_CANDIDATES.values())
    position = 0
    for family, candidates in CLASSIC_CANDIDATES.items():
        print(f'\n▶ FAMILY {family} — {len(candidates)} kandidat', flush=True)
        for cfg in candidates:
            position += 1
            record, probability = evaluate_classic_candidate(family, cfg, position, total_candidates)
            if record is not None:
                CLASSIC_ROWS.append(record)
                CLASSIC_PROB_BY_SIGNATURE[record['signature']] = probability

    if not CLASSIC_ROWS:
        raise RuntimeError('Semua kandidat model klasik gagal.')
    CLASSIC_LEADERBOARD = pd.DataFrame(CLASSIC_ROWS).sort_values(
        ['roc_auc', 'f1_macro', 'balanced_accuracy'], ascending=[False, False, False]
    ).reset_index(drop=True)
    CLASSIC_LEADERBOARD.to_csv(RUN_DIR / 'binary_classic_development_leaderboard.csv', index=False)

    CLASSIC_BEST = {}
    OOF_PROBS = {}
    MODEL_SPECS = {}
    DEV_MODEL_INFO = {}
    for family in CLASSIC_LEADERBOARD['family'].unique():
        row = CLASSIC_LEADERBOARD[CLASSIC_LEADERBOARD.family == family].iloc[0].to_dict()
        record = next(r for r in CLASSIC_ROWS if r['signature'] == row['signature'])
        CLASSIC_BEST[family] = record
        OOF_PROBS[family] = CLASSIC_PROB_BY_SIGNATURE[record['signature']]
        MODEL_SPECS[family] = {'kind': 'classic', 'family': family, 'config': record['config']}
        DEV_MODEL_INFO[family] = {k: record[k] for k in binary_metrics(Y_DEV, OOF_PROBS[family], record['threshold'])}

    # III.D, hlm. 3: prior seluruh development; baseline deskriptif, bukan kandidat pemenang/ensemble.
    prior_prob = np.full(len(Y_DEV), float(Y_DEV.mean()))
    prior_threshold = 0.5
    OOF_PROBS['DummyPrior'] = prior_prob
    MODEL_SPECS['DummyPrior'] = {'kind': 'dummy'}
    DEV_MODEL_INFO['DummyPrior'] = binary_metrics(Y_DEV, prior_prob, prior_threshold)

    summary = pd.DataFrame([
        {'model': name, **metrics} for name, metrics in DEV_MODEL_INFO.items()
    ]).sort_values(['roc_auc', 'f1_macro'], ascending=False)
    summary.to_csv(RUN_DIR / 'binary_family_best_development.csv', index=False)
    (RUN_DIR / 'binary_classic_best.json').write_text(json.dumps(CLASSIC_BEST, indent=2), encoding='utf-8')
    print('\nModel klasik terbaik per family (development OOF):')
    display(summary.round(4))
    print('Locked test belum disentuh.')
except KeyboardInterrupt:
    print('⏸ SEL 5/11 DIHENTIKAN OLEH PENGGUNA — Pencarian model klasik pada development set', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 5/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 5/11 SELESAI — Pencarian model klasik pada development set ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 6. Arsitektur neural dan pemilihan epoch tanpa memakai holdout

**Rujukan paper:** III.E, halaman 3; III.H, halaman 4; IV.D dan Gambar 5, halaman 6–7.

MLP menerima 11 fitur melalui dua atau tiga hidden layer dengan linear transformation, LayerNorm, GELU, dan dropout. Transformer mengubah fitur numerik menjadi token $t_j=x_jw_j+b_j$, menambahkan classification token, lalu memprosesnya dengan blok attention dan feed-forward. Implementasi ini terinspirasi FT-Transformer, bukan reproduksi persis arsitektur aslinya.

Kedua model menghasilkan satu logit dan dilatih memakai `BCEWithLogitsLoss`, AdamW, gradient clipping 1.0, serta cosine learning-rate schedule. Probabilitas diperoleh melalui sigmoid $\sigma(z)$. Bobot kelas positif adalah $w_+=(N_-/N_+)^\alpha$, dengan $\alpha=0$ atau $0.5$ menurut kandidat dan jumlah kelas dihitung dari subset fitting. **Notasi:** $\sigma$ menyatakan sigmoid; $\alpha$ menyatakan pangkat pembobotan kelas, bukan fungsi sigmoid.

Untuk setiap outer fold, 18% bagian training disisihkan sebagai inner-validation terstratifikasi. Scaler pertama dipelajari dari inner-training; model awal memilih epoch berdasarkan inner-validation loss, maksimum 320 epoch, dengan patience 35 dan minimum 25 epoch. Model pemilih epoch kemudian diganti model baru, dan scaler dipelajari ulang pada seluruh outer-training. Model baru dilatih selama epoch terpilih sebelum memprediksi outer-validation.

Dengan demikian, outer-validation tidak menentukan durasi pelatihannya sendiri. Horizon cosine scheduler tetap 320 bahkan pada refit yang lebih pendek. Riwayat inner-training/validation loss disimpan untuk Gambar 5; kurva tersebut bukan kurva loss pada holdout. Sel ini mendefinisikan arsitektur dan trainer; pencarian kandidat dijalankan di Sel 7.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 6/11 DIMULAI — Menyiapkan MLP dan FT-Transformer', flush=True)
try:
    if not TORCH_AVAILABLE:
        print('⚠ PyTorch tidak tersedia; seluruh model neural akan dilewati.')
        NEURAL_CONFIGS = {}
    else:
        class BinaryMLP(nn.Module):
            def __init__(self, d_in, hidden=(128, 64), dropout=.2):
                super().__init__()
                layers, d = [], d_in
                for width in hidden:
                    layers += [nn.Linear(d, width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(dropout)]
                    d = width
                layers.append(nn.Linear(d, 1))
                self.net = nn.Sequential(*layers)
                for module in self.modules():
                    if isinstance(module, nn.Linear):
                        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                        nn.init.zeros_(module.bias)
            def forward(self, x): return self.net(x).squeeze(-1)

        # III.E, hlm. 3: token fitur numerik t_j = x_j * w_j + b_j, dengan parameter per fitur.
        class NumericTokenizer(nn.Module):
            def __init__(self, n_features, d_token):
                super().__init__()
                self.weight = nn.Parameter(torch.empty(n_features, d_token))
                self.bias = nn.Parameter(torch.empty(n_features, d_token))
                bound = 1 / math.sqrt(d_token)
                nn.init.uniform_(self.weight, -bound, bound)
                nn.init.uniform_(self.bias, -bound, bound)
            def forward(self, x): return x.unsqueeze(-1) * self.weight + self.bias

        # III.E, hlm. 3: transformer tabular terinspirasi FT-Transformer, bukan reproduksi persis.
        class BinaryFTTransformer(nn.Module):
            def __init__(self, d_in, d_token=32, blocks=2, heads=4, ffn_mult=2.0, dropout=.1):
                super().__init__()
                if d_token % heads:
                    raise ValueError('d_token harus habis dibagi heads.')
                self.tokenizer = NumericTokenizer(d_in, d_token)
                self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
                nn.init.normal_(self.cls, std=.02)
                layer = nn.TransformerEncoderLayer(
                    d_model=d_token, nhead=heads, dim_feedforward=int(d_token*ffn_mult),
                    dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
                )
                self.encoder = nn.TransformerEncoder(
                    layer, num_layers=blocks, enable_nested_tensor=False)
                self.head = nn.Sequential(nn.LayerNorm(d_token), nn.GELU(), nn.Linear(d_token, 1))
            def forward(self, x):
                tokens = self.tokenizer(x)
                tokens = torch.cat([self.cls.expand(x.shape[0], -1, -1), tokens], dim=1)
                return self.head(self.encoder(tokens)[:, 0]).squeeze(-1)

        def neural_scaler(name, n_rows):
            if name == 'standard': return StandardScaler()
            if name == 'robust': return RobustScaler(quantile_range=(10, 90))
            if name == 'power': return PowerTransformer(method='yeo-johnson', standardize=True)
            if name == 'quantile':
                return QuantileTransformer(n_quantiles=min(256, n_rows), output_distribution='normal', random_state=SEED)
            raise ValueError(name)

        def make_neural_model(cfg, seed):
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
            if cfg['family'] == 'MLP':
                return BinaryMLP(len(FEATURE_COLUMNS), tuple(cfg['hidden']), cfg['dropout'])
            return BinaryFTTransformer(len(FEATURE_COLUMNS), cfg['d_token'], cfg['blocks'],
                                       cfg['heads'], cfg['ffn_mult'], cfg['dropout'])

        def _to_tensor(values, dtype=torch.float32):
            return torch.as_tensor(np.ascontiguousarray(values), dtype=dtype, device=DEVICE)

        @torch.no_grad()
        def neural_probability(model, X, batch_size=4096):
            model.eval(); result = []
            tensor = _to_tensor(X)
            for start in range(0, len(tensor), batch_size):
                result.append(torch.sigmoid(model(tensor[start:start+batch_size])).cpu().numpy())
            return np.concatenate(result)

        def train_neural(X_train, y_train, cfg, seed, X_stop=None, y_stop=None, fixed_epochs=None, progress_prefix=''):
            model = make_neural_model(cfg, seed).to(DEVICE)
            xt, yt = _to_tensor(X_train), _to_tensor(y_train.astype(np.float32))
            xv = yv = None
            if X_stop is not None:
                xv, yv = _to_tensor(X_stop), _to_tensor(y_stop.astype(np.float32))
            pos = max(float(y_train.sum()), 1.0); neg = max(float(len(y_train)-y_train.sum()), 1.0)
            pos_weight = torch.tensor([(neg/pos) ** float(cfg.get('weight_power', 0.0))], device=DEVICE)
            # III.E, hlm. 3: loss menerima logit; sigmoid dipakai saat menghasilkan probabilitas.
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
            epochs = int(fixed_epochs or cfg.get('epochs', NEURAL_MAX_EPOCHS))
            schedule_horizon = max(int(cfg.get('epochs', NEURAL_MAX_EPOCHS)), 1)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=schedule_horizon, eta_min=cfg['lr']*.02)
            batch_size = int(cfg['batch_size'])
            best_loss, best_epoch, bad, best_state = float('inf'), epochs, 0, None
            history = []
            for epoch in range(1, epochs+1):
                model.train(); perm = torch.randperm(len(xt), device=DEVICE); running = 0.0
                for start in range(0, len(xt), batch_size):
                    idx = perm[start:start+batch_size]
                    optimizer.zero_grad(set_to_none=True)
                    loss = loss_fn(model(xt[idx]), yt[idx])
                    if not torch.isfinite(loss): raise FloatingPointError('Neural loss non-finite.')
                    loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step(); running += loss.item() * len(idx)
                scheduler.step()
                train_loss = running / len(xt)
                val_loss = np.nan
                if xv is not None:
                    model.eval()
                    with torch.no_grad(): val_loss = float(loss_fn(model(xv), yv).item())
                    if val_loss < best_loss - 1e-5:
                        best_loss, best_epoch, bad = val_loss, epoch, 0
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    else:
                        bad += 1
                history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
                if xv is not None and bad >= int(cfg.get('patience', NEURAL_PATIENCE)) and epoch >= 25:
                    break
                if epoch % 50 == 0:
                    print(f'      {progress_prefix} epoch {epoch}/{epochs} | train={train_loss:.4f} | val={val_loss:.4f}', flush=True)
            if xv is not None and best_state is not None:
                model.load_state_dict(best_state)
            else:
                best_epoch = epochs
            return model, int(best_epoch), pd.DataFrame(history)

        base_neural = [
            {'family':'MLP','hidden':[32,16],'dropout':.10,'lr':1e-3,'weight_decay':1e-4,'batch_size':64,'scaler':'standard','weight_power':0.0},
            {'family':'MLP','hidden':[64,32],'dropout':.15,'lr':1e-3,'weight_decay':1e-4,'batch_size':64,'scaler':'power','weight_power':0.0},
            {'family':'MLP','hidden':[128,64],'dropout':.20,'lr':7e-4,'weight_decay':1e-4,'batch_size':64,'scaler':'standard','weight_power':0.5},
            {'family':'MLP','hidden':[256,128,64],'dropout':.25,'lr':5e-4,'weight_decay':3e-4,'batch_size':64,'scaler':'quantile','weight_power':0.0},
            {'family':'MLP','hidden':[128,64,32],'dropout':.10,'lr':1.5e-3,'weight_decay':1e-5,'batch_size':128,'scaler':'power','weight_power':0.5},
            {'family':'MLP','hidden':[64,64,32],'dropout':.30,'lr':7e-4,'weight_decay':1e-3,'batch_size':64,'scaler':'robust','weight_power':0.0},
            {'family':'MLP','hidden':[256,128],'dropout':.15,'lr':1e-3,'weight_decay':1e-4,'batch_size':128,'scaler':'standard','weight_power':0.0},
            {'family':'MLP','hidden':[64,32,16],'dropout':.05,'lr':2e-3,'weight_decay':1e-5,'batch_size':64,'scaler':'quantile','weight_power':0.5},
            {'family':'FTTransformer','d_token':16,'blocks':2,'heads':4,'ffn_mult':2.0,'dropout':.10,'lr':7e-4,'weight_decay':1e-5,'batch_size':64,'scaler':'standard','weight_power':0.0},
            {'family':'FTTransformer','d_token':32,'blocks':2,'heads':4,'ffn_mult':2.0,'dropout':.10,'lr':5e-4,'weight_decay':1e-5,'batch_size':64,'scaler':'power','weight_power':0.0},
            {'family':'FTTransformer','d_token':32,'blocks':3,'heads':4,'ffn_mult':3.0,'dropout':.15,'lr':5e-4,'weight_decay':1e-4,'batch_size':64,'scaler':'standard','weight_power':0.5},
            {'family':'FTTransformer','d_token':64,'blocks':2,'heads':8,'ffn_mult':2.0,'dropout':.15,'lr':3e-4,'weight_decay':1e-4,'batch_size':64,'scaler':'quantile','weight_power':0.0},
            {'family':'FTTransformer','d_token':16,'blocks':3,'heads':4,'ffn_mult':4.0,'dropout':.05,'lr':1e-3,'weight_decay':1e-5,'batch_size':128,'scaler':'power','weight_power':0.5},
            {'family':'FTTransformer','d_token':48,'blocks':2,'heads':4,'ffn_mult':2.0,'dropout':.20,'lr':5e-4,'weight_decay':3e-4,'batch_size':64,'scaler':'robust','weight_power':0.0},
            {'family':'FTTransformer','d_token':32,'blocks':4,'heads':8,'ffn_mult':2.0,'dropout':.20,'lr':3e-4,'weight_decay':1e-4,'batch_size':64,'scaler':'standard','weight_power':0.0},
            {'family':'FTTransformer','d_token':64,'blocks':3,'heads':8,'ffn_mult':3.0,'dropout':.10,'lr':3e-4,'weight_decay':1e-5,'batch_size':128,'scaler':'power','weight_power':0.5},
        ]
        NEURAL_CONFIGS = {
            family: [dict(cfg, epochs=NEURAL_MAX_EPOCHS, patience=NEURAL_PATIENCE)
                     for cfg in base_neural if cfg['family']==family][:NEURAL_LIMIT_PER_FAMILY]
            for family in ['MLP','FTTransformer']
        }

        # III.E, hlm. 3: pilih epoch pada inner-validation, lalu refit sebelum prediksi outer fold.
        def evaluate_neural_candidate(cfg, position, total):
            signature = hashlib.sha256((EVALUATION_CONTEXT_HASH+'neural'+json.dumps(json_safe(cfg),sort_keys=True)).encode()).hexdigest()[:16]
            stem = f'{slug(cfg["family"])}_{signature}'
            npz_path = RUN_DIR/'cache'/'binary_neural'/f'{stem}.npz'
            json_path = RUN_DIR/'cache'/'binary_neural'/f'{stem}.json'
            history_path = RUN_DIR/'cache'/'binary_neural'/f'{stem}_inner_history.csv'
            if npz_path.exists() and json_path.exists():
                try:
                    record = json.loads(json_path.read_text(encoding='utf-8'))
                    with np.load(npz_path) as packed:
                        probability = packed['probability'].copy()
                    if (record.get('signature') == signature and len(probability) == len(Y_DEV)
                            and np.isfinite(probability).all() and history_path.is_file()
                            and record.get('history_file') == history_path.name):
                        print(f'  ✓ {position}/{total} {cfg["family"]} cache | AUC={record["roc_auc"]:.4f}', flush=True)
                        return record, probability
                    raise ValueError('isi cache tidak cocok')
                except Exception as cache_error:
                    print(f'  ⚠ Cache {cfg["family"]} rusak/tidak cocok; dihitung ulang: {cache_error}', flush=True)
                    npz_path.unlink(missing_ok=True); json_path.unlink(missing_ok=True); history_path.unlink(missing_ok=True)
            print(f'  ▶ {position}/{total} {cfg["family"]} kandidat dimulai', flush=True)
            started = _cell_time.perf_counter(); probability=np.zeros(len(Y_DEV),dtype=float); epochs=[]; inner_histories=[]
            try:
                for fold_no,(outer_train,outer_valid) in enumerate(BINARY_CV_FOLDS,1):
                    inner_fit,inner_stop=train_test_split(
                        outer_train,test_size=INNER_STOP_SIZE,random_state=SEED+1000+fold_no,
                        stratify=Y_DEV[outer_train],shuffle=True)
                    scaler1=neural_scaler(cfg['scaler'],len(inner_fit))
                    xfit=scaler1.fit_transform(X_DEV[inner_fit]).astype(np.float32)
                    xstop=scaler1.transform(X_DEV[inner_stop]).astype(np.float32)
                    selector,best_epoch,selector_history=train_neural(
                        xfit,Y_DEV[inner_fit],cfg,SEED+fold_no,
                        xstop,Y_DEV[inner_stop],progress_prefix=f'fold {fold_no} select')
                    selector_history=selector_history.assign(fold=fold_no,selected_epoch=best_epoch)
                    inner_histories.append(selector_history)
                    del selector; gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
                    scaler2=neural_scaler(cfg['scaler'],len(outer_train))
                    xtrain=scaler2.fit_transform(X_DEV[outer_train]).astype(np.float32)
                    xvalid=scaler2.transform(X_DEV[outer_valid]).astype(np.float32)
                    refit,_,_=train_neural(xtrain,Y_DEV[outer_train],cfg,SEED+100+fold_no,
                                           fixed_epochs=max(1,best_epoch),progress_prefix=f'fold {fold_no} refit')
                    probability[outer_valid]=neural_probability(refit,xvalid)
                    epochs.append(best_epoch)
                    del refit; gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
                    print(f'    ✅ fold {fold_no}/{CV_SPLITS} | selected_epoch={best_epoch}',flush=True)
                threshold=choose_threshold(Y_DEV,probability)
                record={'family':cfg['family'],'config':json_safe(cfg),'selected_epochs':epochs,
                        'history_file':history_path.name,
                        **binary_metrics(Y_DEV,probability,threshold),
                        'fit_seconds':_cell_time.perf_counter()-started,'signature':signature,'status':'ok'}
                atomic_savez(npz_path,probability=probability,selected_epochs=np.asarray(epochs))
                atomic_write_csv(pd.concat(inner_histories,ignore_index=True),history_path)
                atomic_write_json(json_path,record)
                print(f'  ✅ {cfg["family"]} | AUC={record["roc_auc"]:.4f} | macro-F1={record["f1_macro"]:.4f} | {record["fit_seconds"]:.1f}s',flush=True)
                return record,probability
            except Exception as exc:
                history_path.unlink(missing_ok=True)
                failure={'family':cfg['family'],'config':json_safe(cfg),'status':'failed','error_type':type(exc).__name__,'error':str(exc)}
                atomic_write_json(json_path,failure)
                print(f'  ⚠ {cfg["family"]} gagal dan dilewati: {type(exc).__name__}: {exc}',flush=True)
                return None,None

        print(f'Neural device: {DEVICE}')
        for family, configs in NEURAL_CONFIGS.items():
            sample_model=make_neural_model(configs[0],SEED)
            print(f'{family:16s}: {len(configs)} kandidat × {CV_SPLITS} fold | contoh parameter={sum(p.numel() for p in sample_model.parameters()):,}')
            del sample_model
except KeyboardInterrupt:
    print('⏸ SEL 6/11 DIHENTIKAN OLEH PENGGUNA — Menyiapkan MLP dan FT-Transformer', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 6/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 6/11 SELESAI — Menyiapkan MLP dan FT-Transformer ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 7. Seleksi konfigurasi MLP dan transformer dari prediksi OOF

**Rujukan paper:** III.E–F, halaman 3; IV.A, halaman 5; Tabel III dan IV.D, halaman 6–7.

Delapan konfigurasi MLP dan delapan konfigurasi transformer dievaluasi menggunakan fold biner yang sama dengan model klasik. Setiap evaluasi mengikuti prosedur inner-validation dan refit Sel 6, sehingga prediksi OOF berasal dari outer-validation yang tidak dipakai untuk fitting maupun pemilihan epoch model tersebut.

Konfigurasi terbaik per keluarga dipilih menurut ROC-AUC OOF, diikuti macro-F1 dan balanced accuracy jika seri. Keluaran `binary_neural_development_leaderboard.csv` merekam hasil kandidat; probabilitas OOF, konfigurasi terpilih, threshold development, dan lokasi riwayat pelatihan ditambahkan ke struktur yang digunakan Sel 8.

Paper membedakan keberhasilan seleksi development dari generalisasi holdout. Karena konfigurasi dipilih berdasarkan skor OOF yang sama, skor di sini tetap merupakan diagnostik seleksi. Kesimpulan mengenai perbandingan neural dan klasik harus membaca hasil holdout pada Tabel III serta keterbatasan IV.H, bukan hanya memilih angka terbaik pada leaderboard ini. Sel ini belum mengevaluasi holdout.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 7/11 DIMULAI — Pencarian MLP dan FT-Transformer', flush=True)
try:
    # III.E/IV.A, hlm. 3/5: kumpulkan hasil OOF untuk seleksi delapan kandidat per keluarga neural.
    NEURAL_ROWS=[]
    NEURAL_PROB_BY_SIGNATURE={}
    if not RUN_NEURAL or not TORCH_AVAILABLE:
        print('⚠ Pencarian neural dilewati sesuai konfigurasi/ketersediaan PyTorch.')
    else:
        total=sum(len(v) for v in NEURAL_CONFIGS.values()); position=0
        for family,configs in NEURAL_CONFIGS.items():
            print(f'\n▶ FAMILY {family} — {len(configs)} kandidat',flush=True)
            for cfg in configs:
                position+=1
                record,probability=evaluate_neural_candidate(cfg,position,total)
                if record is not None:
                    NEURAL_ROWS.append(record); NEURAL_PROB_BY_SIGNATURE[record['signature']]=probability

    if NEURAL_ROWS:
        NEURAL_LEADERBOARD=pd.DataFrame(NEURAL_ROWS).sort_values(
            ['roc_auc','f1_macro','balanced_accuracy'],ascending=[False,False,False]).reset_index(drop=True)
        NEURAL_LEADERBOARD.to_csv(RUN_DIR/'binary_neural_development_leaderboard.csv',index=False)
        for family in NEURAL_LEADERBOARD.family.unique():
            row=NEURAL_LEADERBOARD[NEURAL_LEADERBOARD.family==family].iloc[0].to_dict()
            record=next(r for r in NEURAL_ROWS if r['signature']==row['signature'])
            probability=NEURAL_PROB_BY_SIGNATURE[record['signature']]
            OOF_PROBS[family]=probability
            MODEL_SPECS[family]={'kind':'neural','family':family,'config':record['config'],
                                 'signature':record['signature'],'history_file':record['history_file']}
            DEV_MODEL_INFO[family]=binary_metrics(Y_DEV,probability,record['threshold'])
        display(NEURAL_LEADERBOARD[['family','roc_auc','f1_macro','balanced_accuracy','threshold','fit_seconds']].round(4))
    else:
        print('Tidak ada hasil neural; eksperimen klasik tetap dapat dilanjutkan.')

    pd.DataFrame([{'model':n,**m} for n,m in DEV_MODEL_INFO.items()]).sort_values(
        ['roc_auc','f1_macro'],ascending=False).to_csv(RUN_DIR/'binary_family_best_development.csv',index=False)
    print('Locked test belum disentuh.')
except KeyboardInterrupt:
    print('⏸ SEL 7/11 DIHENTIKAN OLEH PENGGUNA — Pencarian MLP dan FT-Transformer', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 7/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 7/11 SELESAI — Pencarian MLP dan FT-Transformer ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 8. Penguncian model biner, soft-voting, dan interpretasi development

**Rujukan paper:** III.F, halaman 3–4; III.J, halaman 5; Gambar 3, halaman 5; IV.C, halaman 6; IV.F dan Gambar 6, halaman 8.

Lima keluarga non-baseline dengan peringkat development tertinggi menurut ROC-AUC lalu macro-F1 menyediakan probabilitas OOF untuk ensemble. Soft-voting menggunakan $p_{ens}(x)=\sum_m w_m p_m(x)$, dengan bobot nonnegatif dan jumlah satu. Pencarian memeriksa bobot sama serta 800 sampel Dirichlet; sampel acak dipertahankan jika bobot maksimum tidak melebihi 0.90 dan sedikitnya dua bobot mencapai 0.05. Threshold dicari hanya untuk 30 kandidat bobot dengan AUC tertinggi.

Model utama dan runner-up dipilih sebelum evaluasi holdout. Pada hasil yang dilaporkan, keduanya adalah **SoftVoting** dan **ExtraTrees**. `binary_selection_lock.json` menyimpan pilihan beserta threshold, bobot, dan identitas evaluasinya; `binary_selection_table.csv` menyimpan urutan development. Hasil holdout tidak boleh mengganti penamaan model utama atau runner-up.

`development_model_comparison.png` mendasari Gambar 3. Karena konfigurasi, threshold, dan bobot ensemble dipilih dari OOF yang sama, grafik ini menunjukkan hasil seleksi development, bukan estimasi generalisasi independen.

Analisis fitur untuk Gambar 6 memakai **anchor Random Forest yang ditetapkan sebelumnya**, bukan model utama SoftVoting. Permutation importance dihitung pada held-out development fold: satu fitur diacak 20 kali per fold dan penurunan ROC-AUC diringkas lintas lima fold. Error bar menunjukkan simpangan baku antarfold, bukan confidence interval. Grafik `development_permutation_importance.png` menjelaskan ketergantungan prediksi model acuan pada fitur, bukan pengaruh kausal atau kontribusi langsung fitur terhadap ensemble.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 8/11 DIMULAI — Mengunci model utama dan ensemble', flush=True)
try:
    eligible=[n for n in DEV_MODEL_INFO if n!='DummyPrior']
    if not eligible: raise RuntimeError('Tidak ada model valid untuk dikunci.')
    eligible=sorted(eligible,key=lambda n:(DEV_MODEL_INFO[n]['roc_auc'],DEV_MODEL_INFO[n]['f1_macro']),reverse=True)
    members=eligible[:min(5,len(eligible))]
    P=np.column_stack([OOF_PROBS[n] for n in members])

    if len(members)>1:
        rng=np.random.default_rng(SEED+700)
        weight_candidates=[np.ones(len(members))/len(members)]
        draws=rng.dirichlet(np.ones(len(members)),size=ENSEMBLE_WEIGHT_TRIALS)
        weight_candidates.extend(w for w in draws if np.max(w)<=0.90 and np.sum(w>=0.05)>=2)

        # III.F, hlm. 4: saring bobot menurut AUC OOF; cari threshold hanya untuk 30 kandidat teratas.
        pre=[]
        for w in weight_candidates:
            probability=P@w
            pre.append((roc_auc_score(Y_DEV,probability),w,probability))
        pre=sorted(pre,key=lambda x:x[0],reverse=True)[:min(30,len(pre))]
        best=None
        for auc,w,probability in pre:
            threshold=choose_threshold(Y_DEV,probability)
            metrics=binary_metrics(Y_DEV,probability,threshold)
            key=(metrics['roc_auc'],metrics['f1_macro'],metrics['balanced_accuracy'])
            if best is None or key>best[0]: best=(key,w,probability,metrics)
        _,ensemble_weights,ensemble_probability,ensemble_metrics=best
        OOF_PROBS['SoftVoting']=ensemble_probability
        MODEL_SPECS['SoftVoting']={'kind':'ensemble','members':members,'weights':ensemble_weights.tolist()}
        DEV_MODEL_INFO['SoftVoting']=ensemble_metrics
    else:
        print('Soft-voting dilewati: hanya satu model dasar tersedia.')

    selection_table=pd.DataFrame([{'model':n,**m} for n,m in DEV_MODEL_INFO.items()]).sort_values(
        ['roc_auc','f1_macro','balanced_accuracy'],ascending=False).reset_index(drop=True)
    non_dummy=selection_table[selection_table.model!='DummyPrior'].reset_index(drop=True)
    PRIMARY_BINARY=str(non_dummy.iloc[0].model)
    RUNNER_UP_BINARY=str(non_dummy.iloc[1].model) if len(non_dummy)>1 else PRIMARY_BINARY

    selection_lock={
        'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,'split_seed':SEED,
        'selection_data':'development OOF only','primary_metric':PRIMARY_SELECTION_METRIC,
        'primary_binary_model':PRIMARY_BINARY,'runner_up_binary_model':RUNNER_UP_BINARY,
        'model_specs':json_safe(MODEL_SPECS),'development_metrics':json_safe(DEV_MODEL_INFO),
        'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
        'test_accessed':bool((RUN_DIR/'FINAL_COMPLETE.json').exists() or (RUN_DIR/'TEST_ACCESS_STARTED.json').exists()),
    }
    binary_lock_path=RUN_DIR/'binary_selection_lock.json'
    final_state_exists=(RUN_DIR/'FINAL_COMPLETE.json').exists() or (RUN_DIR/'TEST_ACCESS_STARTED.json').exists()
    if final_state_exists:
        if not binary_lock_path.exists():
            raise RuntimeError('Akses test pernah dimulai tetapi binary selection lock hilang. Gunakan run/protocol baru.')
        existing=json.loads(binary_lock_path.read_text(encoding='utf-8'))
        for key in ['primary_binary_model','runner_up_binary_model','model_specs','development_metrics','evaluation_context_hash']:
            if json.dumps(existing.get(key),sort_keys=True)!=json.dumps(selection_lock.get(key),sort_keys=True):
                raise RuntimeError(f'Akses test pernah dimulai tetapi selection lock berubah pada {key}. Gunakan run/protocol baru.')
        selection_lock['test_accessed']=True
    atomic_write_json(binary_lock_path,selection_lock)
    atomic_write_csv(selection_table, RUN_DIR/'binary_selection_table.csv')

    # IV.A/Gambar 3, hlm. 5: tampilkan skor seleksi development; garis baseline mengikuti metrik sumbu.
    chart_data=selection_table[selection_table.model!='DummyPrior'].head(12).copy()
    fig,axes=plt.subplots(1,2,figsize=(15,5.5))
    for ax,metric,title,color in [
        (axes[0],'roc_auc','ROC-AUC selection score (pooled OOF)','#4C78A8'),
        (axes[1],'f1_macro','Macro-F1 selection score (OOF-tuned threshold)','#54A24B'),
    ]:
        ordered=chart_data.sort_values(metric)
        ax.barh(ordered.model,ordered[metric],color=color,edgecolor='white')
        baseline=float(DEV_MODEL_INFO['DummyPrior'][metric])
        ax.axvline(baseline,color='#E45756',linestyle='--',linewidth=1.4,label=f'DummyPrior {baseline:.3f}')
        ax.set(xlabel=metric,title=title,xlim=(0,1)); ax.legend(loc='lower right',fontsize=8)
    fig.suptitle('Development Tuning Scores — Selected Configurations',fontsize=14)
    fig.tight_layout(); fig.savefig(RUN_DIR/'development_model_comparison.png',dpi=180,bbox_inches='tight'); plt.show()
    print('✅ Grafik perbandingan model tersimpan: development_model_comparison.png')

    PERMUTATION_IMPORTANCE_RAW=pd.DataFrame(); PERMUTATION_IMPORTANCE_SUMMARY=pd.DataFrame()
    if RUN_INTERPRETABILITY:
        # III.J/IV.F, hlm. 5/8: gunakan anchor Random Forest yang ditetapkan sebelum melihat skor.
        # Gambar 6 menjelaskan model acuan ini, bukan SoftVoting; konfigurasi tidak dituning pada fold importance.
        interpretation_family='RandomForest'
        interpretation_label='RandomForest_prespecified'
        interpretation_config=dict(CLASSIC_CANDIDATES[interpretation_family][0])
        importance_path=RUN_DIR/'development_permutation_importance_raw.csv'
        expected_columns={'evaluation_context_hash','model','fold','repeat','feature','base_auc','permuted_auc','auc_drop'}
        if importance_path.exists():
            cached=pd.read_csv(importance_path)
            valid=(set(cached.columns)==expected_columns and len(cached)>0 and
                   cached['evaluation_context_hash'].eq(EVALUATION_CONTEXT_HASH).all() and
                   cached['model'].eq(interpretation_label).all() and
                   np.isfinite(cached[['base_auc','permuted_auc','auc_drop']].to_numpy()).all())
            if valid:
                PERMUTATION_IMPORTANCE_RAW=cached
                print(f'✓ Permutation importance cache digunakan: {interpretation_label}',flush=True)
            else:
                importance_path.unlink(missing_ok=True)
        if PERMUTATION_IMPORTANCE_RAW.empty:
            records=[]
            for fold_no,(tr,va) in enumerate(BINARY_CV_FOLDS,1):
                print(f'  ▶ Permutation importance fold {fold_no}/{CV_SPLITS} — {interpretation_family}',flush=True)
                estimator=make_classic_estimator(interpretation_family,interpretation_config)
                estimator.fit(X_DEV[tr],Y_DEV[tr])
                base_probability=estimator.predict_proba(X_DEV[va])[:,1]
                base_auc=roc_auc_score(Y_DEV[va],base_probability)
                rng=np.random.default_rng(SEED+8100+fold_no)
                for repeat in range(1,PERMUTATION_REPEATS+1):
                    for feature_index,feature in enumerate(FEATURE_COLUMNS):
                        permuted=X_DEV[va].copy()
                        permuted[:,feature_index]=rng.permutation(permuted[:,feature_index])
                        permuted_probability=estimator.predict_proba(permuted)[:,1]
                        permuted_auc=roc_auc_score(Y_DEV[va],permuted_probability)
                        records.append({
                            'evaluation_context_hash':EVALUATION_CONTEXT_HASH,'model':interpretation_label,
                            'fold':fold_no,'repeat':repeat,'feature':feature,'base_auc':base_auc,
                            'permuted_auc':permuted_auc,'auc_drop':base_auc-permuted_auc,
                        })
                print(f'  ✅ Permutation importance fold {fold_no}/{CV_SPLITS} selesai',flush=True)
            PERMUTATION_IMPORTANCE_RAW=pd.DataFrame(records)
            atomic_write_csv(PERMUTATION_IMPORTANCE_RAW,importance_path)

        fold_means=(PERMUTATION_IMPORTANCE_RAW.groupby(['feature','fold'],as_index=False)['auc_drop'].mean())
        summary_rows=[]
        for feature,values in fold_means.groupby('feature')['auc_drop']:
            array=values.to_numpy(dtype=float); mean=float(array.mean())
            fold_sd=float(array.std(ddof=1)) if len(array)>1 else 0.0
            summary_rows.append({'feature':feature,'mean_auc_drop':mean,'fold_sd_auc_drop':fold_sd,
                                 'fold_min_auc_drop':float(array.min()),'fold_max_auc_drop':float(array.max()),
                                 'folds':len(array),'repeats_per_fold':PERMUTATION_REPEATS,
                                 'interpretation_model':interpretation_label})
        PERMUTATION_IMPORTANCE_SUMMARY=pd.DataFrame(summary_rows).sort_values('mean_auc_drop',ascending=False)
        atomic_write_csv(PERMUTATION_IMPORTANCE_SUMMARY,RUN_DIR/'development_permutation_importance_summary.csv')
        plot_data=PERMUTATION_IMPORTANCE_SUMMARY.sort_values('mean_auc_drop')
        fig,ax=plt.subplots(figsize=(9,6))
        ax.barh(plot_data.feature,plot_data.mean_auc_drop,xerr=plot_data.fold_sd_auc_drop,
                color='#F2CF5B',edgecolor='#8C6D1F',error_kw={'linewidth':1,'capsize':3})
        ax.axvline(0,color='black',linewidth=.8)
        ax.set(title=f'Permutation Importance — {interpretation_label}',
               xlabel='Penurunan ROC-AUC setelah fitur diacak (mean ± SD deskriptif antar-fold)')
        fig.tight_layout(); fig.savefig(RUN_DIR/'development_permutation_importance.png',dpi=180,bbox_inches='tight'); plt.show()
        print('✅ Permutation importance model prespecified tersimpan; hasil hanya untuk interpretasi, tidak untuk seleksi fitur.')

    print('Model utama biner DIKUNCI:',PRIMARY_BINARY)
    print('Runner-up DIKUNCI        :',RUNNER_UP_BINARY)
    if 'SoftVoting' in MODEL_SPECS:
        print('Soft-voting members      :',dict(zip(members,np.round(ensemble_weights,4))))
    display(selection_table.round(4))
    print('Status holdout:', 'pernah dibuka/final tersimpan; lock dipertahankan' if final_state_exists else 'belum disentuh')
except KeyboardInterrupt:
    print('⏸ SEL 8/11 DIHENTIKAN OLEH PENGGUNA — Mengunci model utama dan ensemble', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 8/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 8/11 SELESAI — Mengunci model utama dan ensemble ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 9. Pemodelan ordinal sebagai analisis tambahan

**Rujukan paper:** III.G–H, halaman 4; IV.E, Tabel V–VI, halaman 7; kelanjutan pembahasan kelas ekstrem, halaman 8.

Target ordinal mempertahankan label `quality` 3–8. Tiga keluarga yang dilaporkan adalah cumulative logistic classification, histogram-based gradient boosting regression, dan Extra Trees regression, masing-masing 14 konfigurasi. Evaluasi menggunakan lima fold yang distratifikasi menurut kualitas asli. Model dipilih dengan **MAE prediksi kontinu terkecil**, lalu quadratic weighted kappa (QWK) terbesar jika seri; pilihan ini terpisah dari pemenang biner.

Model kumulatif melatih lima classifier independen untuk $r_k(x)=P(q>k\mid x)$, $k=3,\ldots,7$. Koreksi cumulative minimum menghasilkan probabilitas yang tidak meningkat terhadap batas kualitas: $\tilde r_k=\min_{j=3,\ldots,k}r_j$. Prediksi kontinu adalah $\hat q_{cont}=3+\sum_{k=3}^{7}\tilde r_k$, sedangkan prediksi kelas adalah $\hat q_{class}=3+\sum_{k=3}^{7}\mathbf{1}(\tilde r_k\geq0.5)$. Model ini tidak memaksakan koefisien bersama seperti model proportional odds.

Untuk regresor tree, MAE dihitung dari keluaran kontinu; metrik kelas memakai pembulatan dengan `np.rint` dan pembatasan nilai 3–8. Selain MAE dan QWK, evaluasi mencatat exact-grade accuracy, within-one-grade accuracy, dan macro-F1 atas keenam kelas. Within-one yang tinggi perlu dibaca bersama confusion matrix karena toleransi satu tingkat dapat menyembunyikan kegagalan mengenali kelas langka.

`ordinal_development_leaderboard.csv` dan `ordinal_selection_lock.json` merekam hasil seleksi. Model utama ordinal pada paper adalah histogram-based gradient boosting berdasarkan MAE development, bukan model yang dipilih ulang dari metrik holdout terbaik.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 9/11 DIMULAI — Seleksi model ordinal pada development set', flush=True)
try:
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

    class CumulativeOrdinalLogit(BaseEstimator):
        def __init__(self,C=1.0,scaler='standard',class_weight=None):
            self.C=C; self.scaler=scaler; self.class_weight=class_weight
        def fit(self,X,y):
            self.scaler_=make_scaler(self.scaler); xs=self.scaler_.fit_transform(X)
            self.thresholds_=np.asarray(ORDINAL_LEVELS[:-1],dtype=int); self.models_=[]
            for threshold in self.thresholds_:
                target=(np.asarray(y)>threshold).astype(int)
                model=LogisticRegression(C=self.C,solver='liblinear',max_iter=6000,
                                         class_weight=self.class_weight,random_state=SEED,
                                         **logistic_regularization_kwargs('l2'))
                model.fit(xs,target); self.models_.append(model)
            return self
        def predict_cumulative(self,X):
            xs=self.scaler_.transform(X)
            probs=np.column_stack([m.predict_proba(xs)[:,1] for m in self.models_])
            return np.minimum.accumulate(probs,axis=1)  # III.G, hlm. 4: cumulative minimum menjaga P(q > k) tidak meningkat terhadap k.
        def predict_expected(self,X): return float(ORDINAL_MIN)+self.predict_cumulative(X).sum(axis=1)
        def predict(self,X): return (ORDINAL_MIN+(self.predict_cumulative(X)>=.5).sum(axis=1)).astype(int)

    def ordinal_metrics(y_true,continuous,predicted=None):
        continuous=np.asarray(continuous,dtype=float)
        pred=np.clip(np.rint(continuous),ORDINAL_MIN,ORDINAL_MAX).astype(int) if predicted is None else np.asarray(predicted,dtype=int)
        pred=np.clip(pred,ORDINAL_MIN,ORDINAL_MAX)
        return {
            'mae':float(mean_absolute_error(y_true,continuous)),
            'qwk':float(cohen_kappa_score(y_true,pred,weights='quadratic')),
            'exact_accuracy':float(accuracy_score(y_true,pred)),
            'within_one_accuracy':float(np.mean(np.abs(np.asarray(y_true)-pred)<=1)),
            'macro_f1':float(f1_score(y_true,pred,labels=np.asarray(ORDINAL_LEVELS),average='macro',zero_division=0)),
        }

    def ordinal_candidate_sets():
        limits=4 if MODE=='quick' else 14
        sets={
            'CumulativeLogit':_candidate_subset(
                [{'C':1.0,'scaler':'standard','class_weight':None},{'C':10.0,'scaler':'power','class_weight':None}],
                {'C':[.01,.03,.1,.3,1.,3.,10.,30.,100.], 'scaler':['standard','robust','power'],
                 'class_weight':[None,'balanced']},limits,SEED+801),
            'HistGBRegressor':_candidate_subset(
                [{'learning_rate':.06,'max_iter':600,'max_leaf_nodes':15,'min_samples_leaf':20,'l2_regularization':.1,'loss':'squared_error'}],
                {'learning_rate':[.02,.04,.06,.1], 'max_iter':[300,600,1000], 'max_leaf_nodes':[7,15,31],
                 'min_samples_leaf':[10,20,30,40], 'l2_regularization':[0.,.01,.1,1.,10.],
                 'loss':['squared_error','absolute_error']},limits,SEED+802),
            'ExtraTreesRegressor':_candidate_subset(
                [{'n_estimators':800,'max_depth':None,'max_features':1.0,'min_samples_leaf':1}],
                {'n_estimators':[400,800,1200], 'max_depth':[None,8,14,24], 'max_features':['sqrt',.5,.8,1.0],
                 'min_samples_leaf':[1,2,3,5]},limits,SEED+803),
        }
        if TABPFN_AVAILABLE:
            sets['TabPFNRegressor'] = _candidate_subset(
                [{'n_estimators': 4}, {'n_estimators': 8}], {'n_estimators': [2, 4, 8]},
                min(2, limits), SEED+806)
        if RUN_OPTIONAL_BOOSTERS and XGBOOST_AVAILABLE:
            sets['XGBRegressor']=_candidate_subset(
                [{'n_estimators':600,'max_depth':3,'learning_rate':.04,'min_child_weight':3,'subsample':.85,'colsample_bytree':.9,'reg_alpha':0.,'reg_lambda':2.}],
                {'n_estimators':[300,600,900],'max_depth':[2,3,4,5],'learning_rate':[.02,.04,.07,.1],
                 'min_child_weight':[1,3,5,10],'subsample':[.7,.85,1.0],'colsample_bytree':[.7,.85,1.0],
                 'reg_alpha':[0.,.01,.1,1.],'reg_lambda':[.5,1.,2.,5.]},limits,SEED+804)
        if RUN_OPTIONAL_BOOSTERS and CATBOOST_AVAILABLE:
            sets['CatBoostRegressor']=_candidate_subset(
                [{'iterations':800,'depth':6,'learning_rate':.04,'l2_leaf_reg':3.,'random_strength':.5,'loss_function':'RMSE'}],
                {'iterations':[400,800,1200],'depth':[4,5,6,7,8],'learning_rate':[.02,.04,.07,.1],
                 'l2_leaf_reg':[1.,3.,5.,10.],'random_strength':[0.,.5,1.],
                 'loss_function':['RMSE','MAE']},limits,SEED+805)
        return sets

    def make_ordinal_estimator(family,cfg):
        cfg=dict(cfg)
        if family=='CumulativeLogit': return CumulativeOrdinalLogit(**cfg)
        if family=='HistGBRegressor': return HistGradientBoostingRegressor(random_state=SEED,early_stopping=False,**cfg)
        if family=='ExtraTreesRegressor': return ExtraTreesRegressor(random_state=SEED,n_jobs=N_THREADS,**cfg)
        if family=='TabPFNRegressor':
            return TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True,
                                   memory_saving_mode=True, random_state=SEED, **cfg)
        if family=='XGBRegressor': return XGBRegressor(tree_method='hist',random_state=SEED,n_jobs=N_THREADS,objective='reg:squarederror',**cfg)
        if family=='CatBoostRegressor': return CatBoostRegressor(verbose=False,allow_writing_files=False,random_seed=SEED,thread_count=N_THREADS,task_type='CPU',**cfg)
        raise ValueError(family)

    def evaluate_ordinal_candidate(family,cfg,position,total):
        signature=hashlib.sha256((EVALUATION_CONTEXT_HASH+'ordinal'+family+json.dumps(json_safe(cfg),sort_keys=True)).encode()).hexdigest()[:16]
        npz_path=RUN_DIR/'cache'/'ordinal'/f'{slug(family)}_{signature}.npz'
        json_path=RUN_DIR/'cache'/'ordinal'/f'{slug(family)}_{signature}.json'
        if npz_path.exists() and json_path.exists():
            try:
                record=json.loads(json_path.read_text(encoding='utf-8'))
                with np.load(npz_path) as packed:
                    continuous=packed['continuous'].copy(); predicted=packed['predicted'].copy()
                if (record.get('signature')==signature and len(continuous)==len(Q_DEV)
                        and len(predicted)==len(Q_DEV) and np.isfinite(continuous).all()):
                    print(f'  ✓ {position}/{total} {family} cache | MAE={record["mae"]:.4f}',flush=True)
                    return record,continuous,predicted
                raise ValueError('isi cache tidak cocok')
            except Exception as cache_error:
                print(f'  ⚠ Cache ordinal {family} rusak/tidak cocok; dihitung ulang: {cache_error}',flush=True)
                npz_path.unlink(missing_ok=True); json_path.unlink(missing_ok=True)
        print(f'  ▶ {position}/{total} {family} kandidat dimulai',flush=True); started=_cell_time.perf_counter()
        continuous=np.zeros(len(Q_DEV),dtype=float); predicted=np.zeros(len(Q_DEV),dtype=int)
        try:
            for fold_no,(tr,va) in enumerate(ORDINAL_CV_FOLDS,1):
                est=make_ordinal_estimator(family,cfg); est.fit(X_DEV[tr],Q_DEV[tr])
                if hasattr(est,'predict_expected'):
                    continuous[va]=est.predict_expected(X_DEV[va]); predicted[va]=est.predict(X_DEV[va])
                else:
                    continuous[va]=est.predict(X_DEV[va]); predicted[va]=np.clip(np.rint(continuous[va]),ORDINAL_MIN,ORDINAL_MAX).astype(int)
            record={'family':family,'config':json_safe(cfg),**ordinal_metrics(Q_DEV,continuous,predicted),
                    'fit_seconds':_cell_time.perf_counter()-started,'signature':signature,'status':'ok'}
            atomic_savez(npz_path,continuous=continuous,predicted=predicted)
            atomic_write_json(json_path,record)
            print(f'  ✅ {family} | MAE={record["mae"]:.4f} | QWK={record["qwk"]:.4f} | {record["fit_seconds"]:.1f}s',flush=True)
            return record,continuous,predicted
        except Exception as exc:
            failure={'family':family,'config':json_safe(cfg),'status':'failed','error_type':type(exc).__name__,'error':str(exc)}
            atomic_write_json(json_path,failure)
            print(f'  ⚠ {family} gagal dan dilewati: {type(exc).__name__}: {exc}',flush=True)
            return None,None,None

    ORDINAL_ROWS=[]; ORDINAL_CONTINUOUS={}; ORDINAL_PREDICTED={}; ORDINAL_BEST={}; PRIMARY_ORDINAL=None
    if not RUN_ORDINAL:
        print('Analisis ordinal dilewati sesuai konfigurasi.')
    else:
        sets=ordinal_candidate_sets(); total=sum(len(v) for v in sets.values()); position=0
        for family,candidates in sets.items():
            print(f'\n▶ ORDINAL FAMILY {family} — {len(candidates)} kandidat',flush=True)
            for cfg in candidates:
                position+=1; record,continuous,predicted=evaluate_ordinal_candidate(family,cfg,position,total)
                if record is not None:
                    ORDINAL_ROWS.append(record); ORDINAL_CONTINUOUS[record['signature']]=continuous; ORDINAL_PREDICTED[record['signature']]=predicted
        if not ORDINAL_ROWS: raise RuntimeError('Semua model ordinal gagal.')
        ORDINAL_LEADERBOARD=pd.DataFrame(ORDINAL_ROWS).sort_values(['mae','qwk'],ascending=[True,False]).reset_index(drop=True)
        for family in ORDINAL_LEADERBOARD.family.unique():
            row=ORDINAL_LEADERBOARD[ORDINAL_LEADERBOARD.family==family].iloc[0]
            ORDINAL_BEST[family]=next(r for r in ORDINAL_ROWS if r['signature']==row['signature'])
        PRIMARY_ORDINAL=str(ORDINAL_LEADERBOARD.iloc[0].family)
        ordinal_lock={'primary_ordinal_model':PRIMARY_ORDINAL,'selection_metric':'MAE development OOF (minimum), QWK tie-break',
                      'best_per_family':json_safe(ORDINAL_BEST),'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
                      'test_accessed':bool((RUN_DIR/'FINAL_COMPLETE.json').exists() or (RUN_DIR/'TEST_ACCESS_STARTED.json').exists())}
        ordinal_lock_path=RUN_DIR/'ordinal_selection_lock.json'
        final_state_exists=(RUN_DIR/'FINAL_COMPLETE.json').exists() or (RUN_DIR/'TEST_ACCESS_STARTED.json').exists()
        if final_state_exists:
            if not ordinal_lock_path.exists():
                raise RuntimeError('Akses test pernah dimulai tetapi ordinal selection lock hilang. Gunakan run/protocol baru.')
            existing=json.loads(ordinal_lock_path.read_text(encoding='utf-8'))
            for key in ['primary_ordinal_model','best_per_family','evaluation_context_hash']:
                if json.dumps(existing.get(key),sort_keys=True)!=json.dumps(ordinal_lock.get(key),sort_keys=True):
                    raise RuntimeError(f'Akses test pernah dimulai tetapi ordinal lock berubah pada {key}. Gunakan run/protocol baru.')
            ordinal_lock['test_accessed']=True
        atomic_write_json(ordinal_lock_path,ordinal_lock)
        ORDINAL_LEADERBOARD.to_csv(RUN_DIR/'ordinal_development_leaderboard.csv',index=False)
        print('\nModel ordinal utama DIKUNCI:',PRIMARY_ORDINAL)
        display(ORDINAL_LEADERBOARD[['family','mae','qwk','exact_accuracy','within_one_accuracy','macro_f1','fit_seconds']].round(4))
    final_state_exists=(RUN_DIR/'FINAL_COMPLETE.json').exists() or (RUN_DIR/'TEST_ACCESS_STARTED.json').exists()
    print('Status holdout:', 'pernah dibuka/final tersimpan; lock dipertahankan' if final_state_exists else 'belum disentuh')
except KeyboardInterrupt:
    print('⏸ SEL 9/11 DIHENTIKAN OLEH PENGGUNA — Seleksi model ordinal pada development set', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 9/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 9/11 SELESAI — Seleksi model ordinal pada development set ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 10. Fitting final dan evaluasi holdout setelah pemeriksaan lock

**Rujukan paper:** III.H, halaman 4; III.K, halaman 5; Tabel III, halaman 6; Tabel V, halaman 7.

Sel ini menjadi titik penggunaan holdout dalam alur pemodelan. Preflight memeriksa kelengkapan binary/ordinal selection lock serta kecocokan data, split, konfigurasi, dan konteks evaluasi. Pada jalur evaluasi baru, `TEST_ACCESS_STARTED.json` ditulis secara atomik sebelum `X_TEST`, `Y_TEST`, dan `Q_TEST` disiapkan. Ketidakcocokan lock atau status run menghentikan proses untuk mencegah hasil yang tercampur.

Model klasik dan ordinal terpilih difit pada seluruh 1087 observasi development. Untuk setiap model neural, split internal development baru sebesar 18% menentukan epoch final; setelah itu scaler dan model baru difit pada seluruh development selama durasi tersebut. Jumlah epoch ini **tidak disalin dari grafik CV** dan tidak dipilih menggunakan fitur, label, atau skor holdout. Threshold biner dan bobot ensemble diterapkan sesuai lock Sel 8.

Evaluasi memakai 272 observasi holdout. `test_metrics_binary.csv` dan `test_metrics_ordinal.csv` menjadi sumber metrik Tabel III dan V, sedangkan `test_predictions_binary.csv` dan `test_predictions_ordinal.csv` menyimpan prediksi per observasi untuk statistik serta confusion matrix Sel 11. Artefak model berada dalam `models`; `FINAL_COMPLETE.json` mencatat kelengkapan dan hash artefak.

Dengan `ALLOW_FINAL_RERUN=False`, run lengkap dimuat dari disk setelah validasi hash, bukan dilatih ulang. Recovery atau penghitungan ulang memerlukan pengaktifan eksplisit dan dicatat sebagai recovery/rerun. Keluaran yang terlihat dalam salinan notebook ini tetap merupakan keluaran arsip sebelum penyesuaian kompatibilitas yang dijelaskan di bagian awal, bukan bukti eksekusi ulang source saat ini.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 10/11 DIMULAI — Training final dan evaluasi locked holdout', flush=True)
try:
    complete_path=RUN_DIR/'FINAL_COMPLETE.json'
    access_path=RUN_DIR/'TEST_ACCESS_STARTED.json'
    binary_lock_path=RUN_DIR/'binary_selection_lock.json'
    ordinal_lock_path=RUN_DIR/'ordinal_selection_lock.json'
    binary_metrics_path=RUN_DIR/'test_metrics_binary.csv'
    ordinal_metrics_path=RUN_DIR/'test_metrics_ordinal.csv'
    binary_predictions_path=RUN_DIR/'test_predictions_binary.csv'
    ordinal_predictions_path=RUN_DIR/'test_predictions_ordinal.csv'
    FINAL_ORDINAL_METRICS=pd.DataFrame()
    FINAL_ORDINAL_FRAME=pd.DataFrame()

    def canonical_json(value):
        return json.dumps(json_safe(value),sort_keys=True,separators=(',',':'))

    def sha256_file(path):
        digest=hashlib.sha256()
        with open(path,'rb') as handle:
            for chunk in iter(lambda:handle.read(1024*1024),b''):
                digest.update(chunk)
        return digest.hexdigest()

    def atomic_to_csv(frame,path):
        tmp=path.with_name(path.name+'.tmp')
        frame.to_csv(tmp,index=False)
        os.replace(tmp,path)

    # III.H/K, hlm. 4/5: validasi pilihan dan identitas run sebelum membentuk array holdout.
    if not binary_lock_path.is_file():
        raise RuntimeError('Binary selection lock belum ada. Jalankan Sel 8 sampai selesai sebelum Sel 10.')
    selection_lock=json.loads(binary_lock_path.read_text(encoding='utf-8'))
    expected_binary_lock={
        'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,'split_seed':SEED,
        'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
    }
    mismatch={k:(selection_lock.get(k),v) for k,v in expected_binary_lock.items() if selection_lock.get(k)!=v}
    if mismatch:
        raise RuntimeError(f'Binary selection lock tidak cocok: {mismatch}')
    if canonical_json(selection_lock.get('model_specs'))!=canonical_json(MODEL_SPECS):
        raise RuntimeError('Model specs aktif berbeda dari binary selection lock.')
    if canonical_json(selection_lock.get('development_metrics'))!=canonical_json(DEV_MODEL_INFO):
        raise RuntimeError('Development metrics/threshold aktif berbeda dari binary selection lock.')
    PRIMARY_BINARY=selection_lock.get('primary_binary_model')
    RUNNER_UP_BINARY=selection_lock.get('runner_up_binary_model')
    for name in [PRIMARY_BINARY,RUNNER_UP_BINARY]:
        if name not in MODEL_SPECS or name not in DEV_MODEL_INFO:
            raise RuntimeError(f'Model terkunci tidak tersedia pada state aktif: {name}')
        threshold=float(DEV_MODEL_INFO[name].get('threshold',np.nan))
        if not np.isfinite(threshold) or not 0.0<=threshold<=1.0:
            raise RuntimeError(f'Threshold development tidak valid untuk {name}: {threshold}')

    olock=None
    if RUN_ORDINAL:
        if not ORDINAL_BEST or not ordinal_lock_path.is_file():
            raise RuntimeError('Ordinal selection lock belum lengkap. Jalankan Sel 9 sampai selesai sebelum Sel 10.')
        olock=json.loads(ordinal_lock_path.read_text(encoding='utf-8'))
        if olock.get('evaluation_context_hash')!=EVALUATION_CONTEXT_HASH:
            raise RuntimeError('Ordinal selection lock tidak cocok dengan evaluation context aktif.')
        if canonical_json(olock.get('best_per_family'))!=canonical_json(ORDINAL_BEST):
            raise RuntimeError('Kandidat ordinal aktif berbeda dari ordinal selection lock.')
        PRIMARY_ORDINAL=olock.get('primary_ordinal_model')
        if PRIMARY_ORDINAL not in ORDINAL_BEST:
            raise RuntimeError(f'Model ordinal terkunci tidak tersedia: {PRIMARY_ORDINAL}')
    else:
        PRIMARY_ORDINAL=None

    completed=None
    if complete_path.exists():
        completed=json.loads(complete_path.read_text(encoding='utf-8'))
        expected_complete={'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,'split_seed':SEED,
                           'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
                           'primary_binary_model':PRIMARY_BINARY,'runner_up_binary_model':RUNNER_UP_BINARY,
                           'primary_ordinal_model':PRIMARY_ORDINAL}
        mismatch={k:(completed.get(k),v) for k,v in expected_complete.items() if completed.get(k)!=v}
        if mismatch:
            raise RuntimeError(f'FINAL_COMPLETE tidak cocok dengan run/lock aktif: {mismatch}')

    access_record=None
    if access_path.exists():
        access_record=json.loads(access_path.read_text(encoding='utf-8'))
        expected_access={'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,
                         'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
                         'primary_binary_model':PRIMARY_BINARY,'runner_up_binary_model':RUNNER_UP_BINARY,
                         'primary_ordinal_model':PRIMARY_ORDINAL}
        mismatch={k:(access_record.get(k),v) for k,v in expected_access.items() if access_record.get(k)!=v}
        if mismatch:
            raise RuntimeError(f'TEST_ACCESS_STARTED tidak cocok dengan run/lock aktif: {mismatch}')
    elif completed is not None:
        raise RuntimeError('FINAL_COMPLETE ada tetapi marker TEST_ACCESS_STARTED hilang; audit trail tidak lengkap.')

    if completed is not None and not ALLOW_FINAL_RERUN:
        print('✅ Evaluasi final sudah selesai; model tidak dilatih dan test tidak diprediksi ulang.',flush=True)
        artifact_hashes=completed.get('artifact_sha256')
        if not isinstance(artifact_hashes,dict) or not artifact_hashes:
            raise RuntimeError('FINAL_COMPLETE tidak memiliki hash artefak.')
        for filename,expected_hash in artifact_hashes.items():
            artifact=RUN_DIR/filename
            if not artifact.is_file():
                raise FileNotFoundError(f'Artefak final hilang: {artifact}')
            actual_hash=sha256_file(artifact)
            if actual_hash!=expected_hash:
                raise RuntimeError(f'Hash artefak final berubah: {filename}')

        # III.K, hlm. 5: pada jalur pemuatan hasil, bentuk label test setelah lock dan hash artefak valid.
        X_TEST=X_ALL[TEST_IDX]
        Y_TEST=Y_BINARY[TEST_IDX]
        Q_TEST=Y_QUALITY[TEST_IDX]
        FINAL_BINARY_METRICS=pd.read_csv(binary_metrics_path)
        FINAL_BINARY_FRAME=pd.read_csv(binary_predictions_path)
        if len(FINAL_BINARY_FRAME)!=len(TEST_IDX) or not np.array_equal(FINAL_BINARY_FRAME['clean_row_index'].to_numpy(dtype=np.int64),TEST_IDX):
            raise RuntimeError('Urutan/identitas baris pada prediksi binary tidak cocok dengan locked test.')
        if (not np.array_equal(FINAL_BINARY_FRAME['quality_true'].to_numpy(dtype=np.int64),Q_TEST) or
                not np.array_equal(FINAL_BINARY_FRAME['binary_true'].to_numpy(dtype=np.int64),Y_TEST)):
            raise RuntimeError('Label pada artefak prediksi binary tidak cocok dengan dataset locked test.')
        expected_models=set(selection_lock['development_metrics'])
        if set(FINAL_BINARY_METRICS['model'])!=expected_models:
            raise RuntimeError('Daftar model pada test_metrics_binary tidak cocok dengan selection lock.')
        for name in expected_models:
            for column in [f'p_{name}',f'pred_{name}']:
                if column not in FINAL_BINARY_FRAME:
                    raise RuntimeError(f'Kolom artefak final hilang: {column}')
            probability=FINAL_BINARY_FRAME[f'p_{name}'].to_numpy(dtype=float)
            threshold=float(selection_lock['development_metrics'][name]['threshold'])
            saved_threshold=float(FINAL_BINARY_METRICS.loc[FINAL_BINARY_METRICS.model==name,'threshold'].iloc[0])
            if not np.isfinite(probability).all() or np.any((probability<0)|(probability>1)):
                raise RuntimeError(f'Probabilitas final tidak valid: {name}')
            if not np.isclose(saved_threshold,threshold,rtol=0,atol=1e-12):
                raise RuntimeError(f'Threshold final berbeda dari development lock: {name}')
            if not np.array_equal(FINAL_BINARY_FRAME[f'pred_{name}'].to_numpy(dtype=int),(probability>=threshold).astype(int)):
                raise RuntimeError(f'Prediksi kelas tidak konsisten dengan threshold: {name}')
        FINAL_BINARY_PROBS={c[2:]:FINAL_BINARY_FRAME[c].to_numpy(dtype=float) for c in FINAL_BINARY_FRAME if c.startswith('p_')}

        if RUN_ORDINAL:
            FINAL_ORDINAL_METRICS=pd.read_csv(ordinal_metrics_path)
            FINAL_ORDINAL_FRAME=pd.read_csv(ordinal_predictions_path)
            if len(FINAL_ORDINAL_FRAME)!=len(TEST_IDX) or not np.array_equal(FINAL_ORDINAL_FRAME['clean_row_index'].to_numpy(dtype=np.int64),TEST_IDX):
                raise RuntimeError('Urutan/identitas baris pada prediksi ordinal tidak cocok dengan locked test.')
            if not np.array_equal(FINAL_ORDINAL_FRAME['quality_true'].to_numpy(dtype=np.int64),Q_TEST):
                raise RuntimeError('Label pada artefak prediksi ordinal tidak cocok dengan dataset locked test.')
            expected_ordinal=set(olock['best_per_family'])
            if set(FINAL_ORDINAL_METRICS['model'])!=expected_ordinal:
                raise RuntimeError('Daftar model pada test_metrics_ordinal tidak cocok dengan ordinal lock.')
            for family in expected_ordinal:
                if f'score_{family}' not in FINAL_ORDINAL_FRAME or f'pred_{family}' not in FINAL_ORDINAL_FRAME:
                    raise RuntimeError(f'Kolom artefak ordinal hilang: {family}')

        selection_lock['test_accessed']=True
        atomic_write_json(binary_lock_path,selection_lock)
        if olock is not None:
            olock['test_accessed']=True
            atomic_write_json(ordinal_lock_path,olock)
        access_record['status']='completed'
        atomic_write_json(access_path,access_record)
        TEST_ACCESSED=True
    else:
        if access_record is not None and completed is None and not ALLOW_FINAL_RERUN:
            raise RuntimeError(
                'Sel 10 pernah dimulai tetapi belum selesai. Periksa penyebab error, lalu set '
                'ALLOW_FINAL_RERUN=True di Sel 1 untuk recovery berlabel; hasil tidak akan diklaim sebagai evaluasi pertama.'
            )

        recovery_mode=access_record is not None or completed is not None
        attempt=(int(access_record.get('attempt',1))+1) if access_record is not None else 1
        access_record={
            'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,
            'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
            'primary_binary_model':PRIMARY_BINARY,'runner_up_binary_model':RUNNER_UP_BINARY,
            'primary_ordinal_model':PRIMARY_ORDINAL,'attempt':attempt,
            'recovery_or_rerun':bool(recovery_mode),'status':'started',
            'started_utc':pd.Timestamp.now(tz='UTC').isoformat(),
        }
        # III.K, hlm. 5: catat awal akses test secara atomik sebelum membuka fitur dan label holdout.
        atomic_write_json(access_path,access_record)
        selection_lock['test_accessed']=True
        atomic_write_json(binary_lock_path,selection_lock)
        if olock is not None:
            olock['test_accessed']=True
            atomic_write_json(ordinal_lock_path,olock)

        X_TEST=X_ALL[TEST_IDX]
        Y_TEST=Y_BINARY[TEST_IDX]
        Q_TEST=Y_QUALITY[TEST_IDX]
        TEST_ACCESSED=True
        label='RECOVERY/RERUN BERLABEL' if recovery_mode else 'EVALUASI FINAL PERTAMA'
        print(f'⚠ Locked holdout dibuka sekarang — {label}.',flush=True)

        FINAL_BINARY_PROBS={}
        base_names=[n for n,spec in MODEL_SPECS.items() if spec['kind'] in {'classic','neural'}]
        for number,name in enumerate(base_names,1):
            spec=MODEL_SPECS[name]
            print(f'▶ Final binary {number}/{len(base_names)} — {name}',flush=True)
            started=_cell_time.perf_counter()
            if spec['kind']=='classic':
                estimator=make_classic_estimator(spec['family'],spec['config'])
                estimator.fit(X_DEV,Y_DEV)
                probability=estimator.predict_proba(X_TEST)[:,1]
                joblib.dump(estimator,RUN_DIR/'models'/f'binary_{slug(name)}.joblib')
            else:
                cfg=spec['config']
                inner_fit,inner_stop=train_test_split(np.arange(len(Y_DEV)),test_size=INNER_STOP_SIZE,
                                                       random_state=SEED+9001,stratify=Y_DEV,shuffle=True)
                scaler1=neural_scaler(cfg['scaler'],len(inner_fit))
                xfit=scaler1.fit_transform(X_DEV[inner_fit]).astype(np.float32)
                xstop=scaler1.transform(X_DEV[inner_stop]).astype(np.float32)
                selector,best_epoch,history=train_neural(xfit,Y_DEV[inner_fit],cfg,SEED+9002,
                                                           xstop,Y_DEV[inner_stop],progress_prefix=f'{name} select')
                del selector; gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                scaler2=neural_scaler(cfg['scaler'],len(Y_DEV))
                xdev=scaler2.fit_transform(X_DEV).astype(np.float32)
                xtest=scaler2.transform(X_TEST).astype(np.float32)
                final_model,_,_=train_neural(xdev,Y_DEV,cfg,SEED+9003,fixed_epochs=max(1,best_epoch),progress_prefix=f'{name} refit')
                probability=neural_probability(final_model,xtest)
                torch.save(final_model.state_dict(),RUN_DIR/'models'/f'binary_{slug(name)}_weights.pt')
                joblib.dump(scaler2,RUN_DIR/'models'/f'binary_{slug(name)}_scaler.joblib')
                history.to_csv(RUN_DIR/'models'/f'binary_{slug(name)}_epoch_selection.csv',index=False)
                atomic_write_json(RUN_DIR/'models'/f'binary_{slug(name)}_metadata.json',
                                  {'config':cfg,'selected_epoch':best_epoch,'seed_select':SEED+9002,'seed_refit':SEED+9003})
            FINAL_BINARY_PROBS[name]=np.asarray(probability,dtype=float)
            print(f'✅ Final {name} selesai ({_cell_time.perf_counter()-started:.1f}s)',flush=True)

        if 'SoftVoting' in MODEL_SPECS:
            spec=MODEL_SPECS['SoftVoting']; weights=np.asarray(spec['weights'],dtype=float)
            missing=[m for m in spec['members'] if m not in FINAL_BINARY_PROBS]
            if missing: raise RuntimeError(f'Anggota ensemble gagal tersedia: {missing}')
            FINAL_BINARY_PROBS['SoftVoting']=sum(w*FINAL_BINARY_PROBS[m] for m,w in zip(spec['members'],weights))
        FINAL_BINARY_PROBS['DummyPrior']=np.full(len(Y_TEST),float(Y_DEV.mean()))

        final_binary_rows=[]
        pred_frame=pd.DataFrame({'clean_row_index':TEST_IDX,'quality_true':Q_TEST,'binary_true':Y_TEST})
        for name,probability in FINAL_BINARY_PROBS.items():
            threshold=float(DEV_MODEL_INFO[name]['threshold'])
            metrics=binary_metrics(Y_TEST,probability,threshold)
            final_binary_rows.append({'model':name,**metrics})
            pred_frame[f'p_{name}']=probability
            pred_frame[f'pred_{name}']=(probability>=threshold).astype(int)
        FINAL_BINARY_METRICS=pd.DataFrame(final_binary_rows).sort_values(['roc_auc','f1_macro'],ascending=False)
        FINAL_BINARY_FRAME=pred_frame
        atomic_to_csv(FINAL_BINARY_METRICS,binary_metrics_path)
        atomic_to_csv(pred_frame,binary_predictions_path)

        if RUN_ORDINAL:
            ordinal_rows=[]; ordinal_frame=pd.DataFrame({'clean_row_index':TEST_IDX,'quality_true':Q_TEST})
            for number,(family,record) in enumerate(ORDINAL_BEST.items(),1):
                print(f'▶ Final ordinal {number}/{len(ORDINAL_BEST)} — {family}',flush=True)
                estimator=make_ordinal_estimator(family,record['config']); estimator.fit(X_DEV,Q_DEV)
                if hasattr(estimator,'predict_expected'):
                    continuous=estimator.predict_expected(X_TEST); predicted=estimator.predict(X_TEST)
                else:
                    continuous=estimator.predict(X_TEST); predicted=np.clip(np.rint(continuous),ORDINAL_MIN,ORDINAL_MAX).astype(int)
                metrics=ordinal_metrics(Q_TEST,continuous,predicted)
                ordinal_rows.append({'model':family,**metrics})
                ordinal_frame[f'score_{family}']=continuous; ordinal_frame[f'pred_{family}']=predicted
                joblib.dump(estimator,RUN_DIR/'models'/f'ordinal_{slug(family)}.joblib')
            FINAL_ORDINAL_METRICS=pd.DataFrame(ordinal_rows).sort_values(['mae','qwk'],ascending=[True,False])
            FINAL_ORDINAL_FRAME=ordinal_frame
            atomic_to_csv(FINAL_ORDINAL_METRICS,ordinal_metrics_path)
            atomic_to_csv(ordinal_frame,ordinal_predictions_path)

        artifacts=[binary_metrics_path,binary_predictions_path]
        if RUN_ORDINAL:
            artifacts.extend([ordinal_metrics_path,ordinal_predictions_path])
        artifact_hashes={p.name:sha256_file(p) for p in artifacts}
        complete={'protocol':PROTOCOL_VERSION,'clean_sha256':CLEAN_SHA256,'split_seed':SEED,
                  'evaluation_context_hash':EVALUATION_CONTEXT_HASH,
                  'primary_binary_model':PRIMARY_BINARY,'runner_up_binary_model':RUNNER_UP_BINARY,
                  'primary_ordinal_model':PRIMARY_ORDINAL,
                  'test_rows':int(len(Y_TEST)),'completed_utc':pd.Timestamp.now(tz='UTC').isoformat(),
                  'evaluation_label':'recovery_or_rerun' if recovery_mode else 'first_final_evaluation',
                  'test_access_attempt':attempt,'artifact_sha256':artifact_hashes,
                  'rerun_policy':'Load stored results when ALLOW_FINAL_RERUN=False.'}
        atomic_write_json(complete_path,complete)
        access_record['status']='completed'; access_record['completed_utc']=complete['completed_utc']
        atomic_write_json(access_path,access_record)

    print('\nHASIL LOCKED HOLDOUT — BINARY')
    display(FINAL_BINARY_METRICS.round(4))
    if not FINAL_ORDINAL_METRICS.empty:
        print('\nHASIL LOCKED HOLDOUT — ORDINAL')
        display(FINAL_ORDINAL_METRICS.round(4))
    print('Model utama tetap mengikuti development lock:',PRIMARY_BINARY)
except KeyboardInterrupt:
    print('⏸ SEL 10/11 DIHENTIKAN OLEH PENGGUNA — Training final dan evaluasi locked holdout', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 10/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 10/11 SELESAI — Training final dan evaluasi locked holdout ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)


## 11. Ketidakpastian statistik, diagnostik, dan pembacaan hasil paper

**Rujukan paper:** III.I–J, halaman 4–5; IV.B–D, Tabel IV, dan Gambar 4, halaman 5–6; Gambar 5, IV.E, dan Tabel VI, halaman 7; IV.H, halaman 8.

Analisis menggunakan prediksi holdout yang telah tersimpan, tanpa melatih ulang model atau menyesuaikan threshold. Wilson interval merangkum ketidakpastian accuracy model utama. Uji McNemar exact dua sisi membandingkan benar/salah prediksi pada observasi yang sama antara model utama dan runner-up yang telah dipilih dari development. Uji ini menilai perbedaan **ketepatan klasifikasi**, bukan perbedaan ROC-AUC.

Sebanyak 3000 bootstrap berpasangan distratifikasi menurut kelas biner. Kedua model memakai indeks resampling yang sama dan threshold tetap; selisih dihitung sebagai **model utama dikurangi runner-up**. Persentil 2.5 dan 97.5 menjadi batas interval pada Tabel IV. Keluaran `paired_bootstrap_primary_vs_runner.csv` menyimpan selisih accuracy sebagai fraksi: misalnya `0.0110` setara dengan **1.10 poin persentase**, bukan kenaikan relatif 1.10%. Ringkasan statistik disimpan dalam `statistical_summary.json`.

Tiga keluaran visual menghubungkan prediksi dan riwayat pelatihan dengan paper:

- `binary_final_curves.png` → Gambar 4: confusion matrix model utama serta kurva ROC/precision–recall. Model kurva dipilih berdasarkan urutan development, bukan dirangking ulang dari holdout.
- `neural_inner_learning_curves.png` → Gambar 5: rerata loss dan rentang antarkuartil antar-inner-fold pada rentang epoch yang tersedia di seluruh fold. Rentang bersama 38 epoch MLP dan 56 epoch transformer yang dilaporkan bukan durasi training final.
- `ordinal_final_confusion.png` → visual pendamping Tabel VI: pola prediksi kualitas dan kesalahan pada kelas langka. Nilai within-one-grade perlu dibaca bersama ketepatan kelas persis serta macro-F1.

Selisih positif yang kecil tidak dengan sendirinya membuktikan keunggulan ensemble. Paper melaporkan McNemar $p=0.7201$ dan interval selisih yang mencakup nol. H1 tentang neural dibahas secara deskriptif; H2 tidak memperoleh dukungan statistik dari perbandingan ini. Hipotesis bersifat eksploratif dan dirumuskan setelah eksperimen, sehingga hasil tidak ditafsirkan sebagai pengujian konfirmatori yang telah dipreregistrasi.


In [ ]:
import time as _cell_time
_cell_started = _cell_time.perf_counter()
print('▶ SEL 11/11 DIMULAI — Membuat laporan final dan uji berpasangan', flush=True)
try:
    from sklearn.metrics import classification_report, RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
    from scipy.stats import binomtest

    if not TEST_ACCESSED or not complete_path.exists():
        raise RuntimeError('Jalankan Sel 10 terlebih dahulu.')

    # III.K, hlm. 5: pulihkan pemetaan kolom probabilitas untuk analisis prediksi tersimpan.
    if not FINAL_BINARY_PROBS:
        FINAL_BINARY_PROBS={c[2:]:FINAL_BINARY_FRAME[c].to_numpy() for c in FINAL_BINARY_FRAME if c.startswith('p_')}

    def wilson_interval(successes,n,z=1.959963984540054):
        p=successes/n; den=1+z*z/n
        center=(p+z*z/(2*n))/den
        half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
        return center-half,center+half

    def predictions_for(name):
        row=FINAL_BINARY_METRICS[FINAL_BINARY_METRICS.model==name].iloc[0]
        return (FINAL_BINARY_PROBS[name]>=float(row.threshold)).astype(int)

    primary_pred=predictions_for(PRIMARY_BINARY)
    runner_pred=predictions_for(RUNNER_UP_BINARY)
    correct_primary=primary_pred==Y_TEST; correct_runner=runner_pred==Y_TEST
    b=int(np.sum(correct_primary & ~correct_runner)); c=int(np.sum(~correct_primary & correct_runner))
    mcnemar_p=float(binomtest(min(b,c),n=b+c,p=.5,alternative='two-sided').pvalue) if b+c else 1.0

    # III.I/Tabel IV, hlm. 4-6: bootstrap berpasangan terstratifikasi; selisih primary minus runner-up.
    rng=np.random.default_rng(SEED+1200); class_indices=[np.flatnonzero(Y_TEST==k) for k in [0,1]]
    deltas=[]
    for _ in range(BOOTSTRAP_REPEATS):
        sample=np.concatenate([rng.choice(idx,size=len(idx),replace=True) for idx in class_indices])
        pa=FINAL_BINARY_PROBS[PRIMARY_BINARY][sample]; pb=FINAL_BINARY_PROBS[RUNNER_UP_BINARY][sample]; ys=Y_TEST[sample]
        ta=float(FINAL_BINARY_METRICS.loc[FINAL_BINARY_METRICS.model==PRIMARY_BINARY,'threshold'].iloc[0])
        tb=float(FINAL_BINARY_METRICS.loc[FINAL_BINARY_METRICS.model==RUNNER_UP_BINARY,'threshold'].iloc[0])
        deltas.append([
            roc_auc_score(ys,pa)-roc_auc_score(ys,pb),
            accuracy_score(ys,pa>=ta)-accuracy_score(ys,pb>=tb),
            f1_score(ys,pa>=ta,average='macro')-f1_score(ys,pb>=tb,average='macro'),
        ])
    deltas=np.asarray(deltas)
    ci=np.quantile(deltas,[.025,.975],axis=0)
    point=np.array([
        roc_auc_score(Y_TEST,FINAL_BINARY_PROBS[PRIMARY_BINARY])-roc_auc_score(Y_TEST,FINAL_BINARY_PROBS[RUNNER_UP_BINARY]),
        accuracy_score(Y_TEST,primary_pred)-accuracy_score(Y_TEST,runner_pred),
        f1_score(Y_TEST,primary_pred,average='macro')-f1_score(Y_TEST,runner_pred,average='macro'),
    ])
    comparison=pd.DataFrame({'metric':['ROC-AUC','Accuracy','Macro-F1'],'delta_primary_minus_runner':point,
                             'ci95_low':ci[0],'ci95_high':ci[1]})
    comparison.to_csv(RUN_DIR/'paired_bootstrap_primary_vs_runner.csv',index=False)

    primary_row=FINAL_BINARY_METRICS[FINAL_BINARY_METRICS.model==PRIMARY_BINARY].iloc[0]
    acc_success=int(np.sum(primary_pred==Y_TEST)); acc_ci=wilson_interval(acc_success,len(Y_TEST))
    stats_summary={'primary':PRIMARY_BINARY,'runner_up':RUNNER_UP_BINARY,'mcnemar_b':b,'mcnemar_c':c,
                   'mcnemar_exact_p':mcnemar_p,'primary_accuracy_wilson95':list(acc_ci),
                   'bootstrap_repeats':BOOTSTRAP_REPEATS}
    (RUN_DIR/'statistical_summary.json').write_text(json.dumps(stats_summary,indent=2),encoding='utf-8')

    print(f'Model utama (dipilih development): {PRIMARY_BINARY}')
    print(f'Accuracy holdout: {primary_row.accuracy:.4f} | Wilson 95% CI [{acc_ci[0]:.4f}, {acc_ci[1]:.4f}]')
    print(f'Runner-up development: {RUNNER_UP_BINARY}')
    print(f'McNemar exact: b={b}, c={c}, p={mcnemar_p:.6f}')
    display(comparison.round(4))
    print('\nClassification report model utama:')
    print(classification_report(Y_TEST,primary_pred,target_names=['negatif (0)','positif (1)'],digits=4,zero_division=0))

    # IV.B/Gambar 4, hlm. 5-6: pilih model kurva dari peringkat development, bukan skor test.
    dev_order=pd.read_csv(RUN_DIR/'binary_selection_table.csv')
    plot_names=[n for n in dev_order.model if n in FINAL_BINARY_PROBS and n!='DummyPrior'][:5]
    fig,axes=plt.subplots(1,3,figsize=(17,4.8))
    ConfusionMatrixDisplay.from_predictions(Y_TEST,primary_pred,display_labels=['<6','≥6'],cmap='Blues',ax=axes[0],colorbar=False)
    axes[0].set_title(f'Confusion matrix — {PRIMARY_BINARY}')
    for name in plot_names:
        RocCurveDisplay.from_predictions(Y_TEST,FINAL_BINARY_PROBS[name],name=name,ax=axes[1])
        PrecisionRecallDisplay.from_predictions(Y_TEST,FINAL_BINARY_PROBS[name],name=name,ax=axes[2])
    axes[1].set_title('ROC locked holdout'); axes[2].set_title('Precision–Recall locked holdout')
    fig.tight_layout(); fig.savefig(RUN_DIR/'binary_final_curves.png',dpi=180,bbox_inches='tight'); plt.show()

    neural_names=[name for name,spec in MODEL_SPECS.items() if spec.get('kind')=='neural']
    if neural_names:
        fig,axes=plt.subplots(1,len(neural_names)+1,figsize=(6*(len(neural_names)+1),4.8),squeeze=False)
        epoch_sets=[]; epoch_labels=[]
        for axis,name in zip(axes[0][:-1],neural_names):
            history_path=RUN_DIR/'cache'/'binary_neural'/MODEL_SPECS[name]['history_file']
            history=pd.read_csv(history_path)
            if history['fold'].nunique()!=CV_SPLITS:
                raise RuntimeError(f'History {name} tidak memuat seluruh {CV_SPLITS} fold.')
            common_horizon=int(history.groupby('fold')['epoch'].max().min())
            common_history=history[history.epoch<=common_horizon].copy()
            aggregate=(common_history.groupby('epoch').agg(
                train_mean=('train_loss','mean'),train_q25=('train_loss',lambda x:x.quantile(.25)),
                train_q75=('train_loss',lambda x:x.quantile(.75)),val_mean=('val_loss','mean'),
                val_q25=('val_loss',lambda x:x.quantile(.25)),val_q75=('val_loss',lambda x:x.quantile(.75)),
                folds_active=('fold','nunique')).reset_index())
            if not aggregate.folds_active.eq(CV_SPLITS).all():
                raise RuntimeError(f'Agregasi learning curve {name} tidak memiliki cohort fold yang tetap.')
            x=aggregate.epoch.to_numpy(dtype=float)
            axis.plot(x,aggregate.train_mean,label='train mean',color='#4C78A8')
            axis.fill_between(x,aggregate.train_q25.to_numpy(dtype=float),aggregate.train_q75.to_numpy(dtype=float),
                              color='#4C78A8',alpha=.18,label='train IQR')
            axis.plot(x,aggregate.val_mean,label='inner-val mean',color='#E45756')
            axis.fill_between(x,aggregate.val_q25.to_numpy(dtype=float),aggregate.val_q75.to_numpy(dtype=float),
                              color='#E45756',alpha=.18,label='inner-val IQR')
            axis.set(title=f'{name}: inner-validation curve (common horizon={common_horizon})',
                     xlabel='Epoch',ylabel='BCE loss')
            axis.legend(fontsize=8)
            epoch_sets.append(history.groupby('fold').selected_epoch.first().to_numpy(dtype=float)); epoch_labels.append(name)
        boxplot_label_arg = ({'tick_labels': epoch_labels}
                             if 'tick_labels' in inspect.signature(axes[0][-1].boxplot).parameters
                             else {'labels': epoch_labels})
        axes[0][-1].boxplot(epoch_sets,showmeans=True,**boxplot_label_arg)
        axes[0][-1].set(title='Selected epoch lintas outer fold',ylabel='Epoch',xlabel='Neural family')
        fig.suptitle('Diagnosis Training Neural — Development Only',fontsize=14)
        fig.tight_layout(); fig.savefig(RUN_DIR/'neural_inner_learning_curves.png',dpi=180,bbox_inches='tight'); plt.show()
        print('✅ Kurva belajar inner-validation tersimpan: neural_inner_learning_curves.png')
    else:
        print('ℹ Kurva belajar neural tidak dibuat karena model neural tidak tersedia.')

    if not FINAL_ORDINAL_METRICS.empty:
        ordinal_primary=PRIMARY_ORDINAL
        ordinal_pred=FINAL_ORDINAL_FRAME[f'pred_{ordinal_primary}'].to_numpy(dtype=int)
        fig,ax=plt.subplots(figsize=(6.5,5.5))
        ConfusionMatrixDisplay.from_predictions(Q_TEST,ordinal_pred,labels=np.asarray(ORDINAL_LEVELS),display_labels=np.asarray(ORDINAL_LEVELS),cmap='Purples',ax=ax,colorbar=False)
        ax.set_title(f'Ordinal confusion matrix — {ordinal_primary}')
        fig.tight_layout(); fig.savefig(RUN_DIR/'ordinal_final_confusion.png',dpi=180,bbox_inches='tight'); plt.show()

    print('\nCATATAN INTERPRETASI:')
    print('- Development plots adalah tuning diagnostics; locked holdout adalah estimasi generalisasi utama.')
    print('- Pemenang tetap model yang dikunci dari development, bukan model dengan test score tertinggi.')
    print('- Accuracy 90% tidak dijanjikan; laporkan seluruh metrik dan confidence interval.')
    print('- Perbandingan dengan paper lain bersifat deskriptif bila split/protokol berbeda.')
    print(f'📁 Seluruh artefak final: {RUN_DIR}')
except KeyboardInterrupt:
    print('⏸ SEL 11/11 DIHENTIKAN OLEH PENGGUNA — Membuat laporan final dan uji berpasangan', flush=True)
    raise
except Exception as _cell_error:
    print(f'❌ SEL 11/11 GAGAL — {type(_cell_error).__name__}: {_cell_error}', flush=True)
    raise
else:
    print(f'✅ SEL 11/11 SELESAI — Membuat laporan final dan uji berpasangan ({_cell_time.perf_counter() - _cell_started:.1f} detik)', flush=True)
